## 1. 필요한 라이브러리 불러오기

먼저 우리가 사용할 도구들을 불러와야 합니다. 각각이 어떤 역할을 하는지 알아볼까요?

os는 컴퓨터의 파일과 폴더를 다루기 위한 기본 도구입니다. pandas는 엑셀 같은 표 형태의 데이터(CSV 파일)를 쉽게 다룰 수 있게 해주는 라이브러리입니다. PIL은 이미지 파일을 열고 처리하는 라이브러리로, PNG, JPG 등 다양한 이미지 형식을 읽을 수 있습니다.

torch는 딥러닝 모델을 만들고 학습시키는 PyTorch의 핵심 부분이고, torch.nn은 신경망의 레이어들을 만드는 도구들이 들어있습니다. torch.optim은 모델이 더 정확해지도록 가중치를 조정하는 최적화 알고리즘들이 들어있고, Dataset과 DataLoader는 우리의 데이터를 모델이 학습하기 좋은 형태로 정리해주는 도구입니다. transforms는 이미지를 모델에 맞게 전처리하는 다양한 함수들이 들어있습니다.

In [16]:
import os, random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


## 2. 이미지 전처리와 재현성 설정하기

이번에는 학습용 전처리와 검증/테스트용 전처리를 분리합니다. 학습 데이터에는 약한 augmentation을 적용해서 모델이 작은 위치 변화에 덜 민감하게 만들고, 검증과 테스트에는 항상 같은 전처리를 적용해 공정하게 평가합니다.


In [17]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 8
LR = 1e-3

train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(7),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


## 3. Dataset 만들기

하나의 Dataset 클래스로 train, validation, test를 모두 처리합니다. train/validation은 이미지와 label을 함께 반환하고, test는 제출용 예측만 필요하므로 이미지만 반환합니다.


In [18]:
class XrayCSV(Dataset):
    def __init__(self, df, root_dir="train", transform=None, has_label=True):
        self.df = df.reset_index(drop=True)
        self.files = self.df["file_name"].tolist()
        self.has_label = has_label
        self.labels = self.df["label"].astype(int).tolist() if has_label else None
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img_path = os.path.join(self.root_dir, self.files[i])
        img = Image.open(img_path).convert("L")
        x = self.transform(img) if self.transform else img

        if self.has_label:
            y = self.labels[i]
            return x, y
        return x


## 4. 테스트 데이터 처리 방식

테스트 데이터는 정답이 없으므로 학습이나 검증에 사용하면 안 됩니다. 이 노트북에서는 test 이미지를 마지막 제출 파일 생성 단계에서만 사용합니다.


In [ ]:
# 테스트 데이터는 정답(label)이 없으므로 has_label=False로 사용합니다.
# 학습/검증 데이터와 같은 Dataset 클래스를 재사용하면 전처리 흐름이 더 일관됩니다.


## 5. Train/Validation 분리와 DataLoader 설정

기존 baseline은 전체 train 데이터로 학습만 하고 검증 성능을 확인하지 않았습니다. 여기서는 stratified split으로 클래스 비율을 유지하면서 train/validation을 나누고, validation accuracy와 confusion matrix를 확인합니다.


In [19]:
df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("전체 학습 데이터:", df.shape)
print("클래스 분포")
print(df["label"].value_counts().sort_index())
print("클래스 비율")
print(df["label"].value_counts(normalize=True).sort_index())

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"],
)

train_ds = XrayCSV(train_df, "train", transform=train_tf, has_label=True)
valid_ds = XrayCSV(valid_df, "train", transform=eval_tf, has_label=True)
test_ds = XrayCSV(test_df, "test", transform=eval_tf, has_label=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0)

print("train:", len(train_ds), "valid:", len(valid_ds), "test:", len(test_ds))


전체 학습 데이터: (5216, 2)
클래스 분포
label
0    1341
1    3875
Name: count, dtype: int64
클래스 비율
label
0    0.257094
1    0.742906
Name: proportion, dtype: float64
train: 4172 valid: 1044 test: 624


## 6. 개선된 SimpleCNN 모델 만들기

기존 CNN보다 convolution layer를 하나 추가하고 BatchNorm과 Dropout을 넣었습니다. BatchNorm은 학습을 안정화하고, Dropout은 validation 성능이 떨어지는 과적합을 줄이기 위한 장치입니다.


In [20]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64 * (IMG_SIZE // 8) * (IMG_SIZE // 8), 2),
        )

    def forward(self, x):
        return self.fc(self.conv(x))


## 7. 학습 설정

클래스 분포가 0보다 1에 많이 치우쳐 있으므로 class weight를 적용합니다. 이렇게 하면 모델이 다수 클래스인 label 1만 예측하는 방향으로 치우치는 문제를 줄일 수 있습니다.


In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

model = SimpleCNN().to(device)

class_counts = train_df["label"].value_counts().sort_index().values
class_weights = len(train_df) / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("class weights:", class_weights.detach().cpu().numpy())

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)


device: mps
class weights: [1.944082  0.6731204]


## 8. 모델 학습과 검증

각 epoch마다 train loss와 validation 성능을 함께 확인합니다. 공식 평가지표는 accuracy이지만, 의료 이미지 분류에서는 recall, precision, F1, confusion matrix도 함께 보는 것이 중요합니다.


In [22]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)

    return running_loss / len(loader.dataset)


def evaluate(model, loader):
    model.eval()
    running_loss = 0.0
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            prob = torch.softmax(logits, dim=1)[:, 1]
            pred = logits.argmax(1)

            running_loss += loss.item() * x.size(0)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())
            y_prob.extend(prob.cpu().numpy())

    metrics = {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }
    return metrics

best_acc = 0.0
best_state = None
history = []

for_ep = range(EPOCHS)
for ep in for_ep:
    train_loss = train_one_epoch(model, train_loader)
    valid_metrics = evaluate(model, valid_loader)
    scheduler.step(valid_metrics["accuracy"])

    history.append({"epoch": ep + 1, "train_loss": train_loss, **valid_metrics})

    if valid_metrics["accuracy"] > best_acc:
        best_acc = valid_metrics["accuracy"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {ep+1:02d} | "
        f"train_loss {train_loss:.4f} | "
        f"valid_loss {valid_metrics['loss']:.4f} | "
        f"acc {valid_metrics['accuracy']:.4f} | "
        f"precision {valid_metrics['precision']:.4f} | "
        f"recall {valid_metrics['recall']:.4f} | "
        f"f1 {valid_metrics['f1']:.4f}"
    )

model.load_state_dict(best_state)
print("Best valid accuracy:", best_acc)


Epoch 01 | train_loss 1.3366 | valid_loss 0.3131 | acc 0.9502 | precision 0.9665 | recall 0.9665 | f1 0.9665
Epoch 02 | train_loss 0.3784 | valid_loss 0.2290 | acc 0.9626 | precision 0.9706 | recall 0.9794 | f1 0.9750
Epoch 03 | train_loss 0.2956 | valid_loss 0.3568 | acc 0.9521 | precision 0.9481 | recall 0.9897 | f1 0.9685
Epoch 04 | train_loss 0.2918 | valid_loss 0.1928 | acc 0.9665 | precision 0.9720 | recall 0.9832 | f1 0.9776
Epoch 05 | train_loss 0.3583 | valid_loss 0.2091 | acc 0.9377 | precision 0.9944 | recall 0.9214 | f1 0.9565
Epoch 06 | train_loss 0.2822 | valid_loss 0.1778 | acc 0.9732 | precision 0.9820 | recall 0.9820 | f1 0.9820
Epoch 07 | train_loss 0.2892 | valid_loss 0.2383 | acc 0.9205 | precision 0.9957 | recall 0.8969 | f1 0.9437
Epoch 08 | train_loss 0.1821 | valid_loss 0.1256 | acc 0.9607 | precision 0.9933 | recall 0.9536 | f1 0.9730
Best valid accuracy: 0.9731800766283525


## 9. Validation 결과 확인

validation set에서 accuracy뿐 아니라 precision, recall, F1, confusion matrix를 확인합니다. 특히 confusion matrix를 보면 정상/폐렴 중 어느 쪽을 더 자주 틀리는지 해석할 수 있습니다.


In [24]:
valid_metrics = evaluate(model, valid_loader)
print("Validation metrics")
print("accuracy :", round(valid_metrics["accuracy"], 4))
print("precision:", round(valid_metrics["precision"], 4))
print("recall   :", round(valid_metrics["recall"], 4))
print("f1       :", round(valid_metrics["f1"], 4))
print("confusion matrix")
print(valid_metrics["confusion_matrix"])


Validation metrics
accuracy : 0.9732
precision: 0.982
recall   : 0.982
f1       : 0.982
confusion matrix
[[254  14]
 [ 14 762]]


## 10. 테스트 데이터 예측 및 제출 파일 만들기

검증에서 가장 accuracy가 높았던 모델 가중치로 test 이미지를 예측하고, sample_submission 형식에 맞춰 제출 파일을 생성합니다.


In [25]:
model.eval()
all_preds = []
all_probs = []

with torch.no_grad():
    for x in test_loader:
        x = x.to(device)
        logits = model(x)
        prob = torch.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(1)

        all_probs.extend(prob.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

submission = pd.read_csv("sample_submission.csv")
submission["label"] = all_preds
submission.head()


,file_name,label
0,test_0001.png,0
1,test_0002.png,0
2,test_0003.png,0
3,test_0004.png,1
4,test_0005.png,1


In [26]:
submission.to_csv("baseline_submit.csv", index=False)
print("저장 완료: baseline_submit.csv")
print(submission["label"].value_counts().sort_index())


저장 완료: baseline_submit.csv
label
0    297
1    327
Name: count, dtype: int64


## 11. 다음 실험: ResNet18 Fine-tuning

SimpleCNN은 직접 만든 작은 CNN이라 구조를 이해하기 쉽지만, 복잡한 X-ray 특징을 충분히 학습하기에는 한계가 있을 수 있습니다. 다음 실험에서는 ImageNet으로 사전학습된 ResNet18을 사용하고, 마지막 분류층과 깊은 feature layer 일부만 fine-tuning합니다.

이 실험의 개선 전략은 2가지입니다.

- 사전학습 모델 사용: 일반 이미지에서 학습한 edge, texture, shape feature를 초기값으로 활용
- 학습 전략 개선: 약한 augmentation, class weight, 일부 layer fine-tuning 적용


In [ ]:
from torchvision import models
from torchvision.models import ResNet18_Weights

resnet_train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(7),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

resnet_eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

resnet_train_ds = XrayCSV(train_df, "train", transform=resnet_train_tf, has_label=True)
resnet_valid_ds = XrayCSV(valid_df, "train", transform=resnet_eval_tf, has_label=True)
resnet_test_ds = XrayCSV(test_df, "test", transform=resnet_eval_tf, has_label=False)

resnet_train_loader = DataLoader(resnet_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
resnet_valid_loader = DataLoader(resnet_valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
resnet_test_loader = DataLoader(resnet_test_ds, batch_size=64, shuffle=False, num_workers=0)


In [ ]:
try:
    weights = ResNet18_Weights.DEFAULT
    resnet_model = models.resnet18(weights=weights)
    print("Loaded pretrained ResNet18 weights")
except Exception as e:
    print("Pretrained weight loading failed. Using random initialization.")
    print(e)
    resnet_model = models.resnet18(weights=None)

# 먼저 전체 backbone을 고정하고, 마지막 block(layer4)과 fc만 학습합니다.
for param in resnet_model.parameters():
    param.requires_grad = False

for param in resnet_model.layer4.parameters():
    param.requires_grad = True

num_features = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_features, 2)
resnet_model = resnet_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, resnet_model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

trainable_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in resnet_model.parameters())
print("trainable params:", trainable_params)
print("total params:", total_params)


In [ ]:
RESNET_EPOCHS = 6
best_acc = 0.0
best_state = None
resnet_history = []

for ep in range(RESNET_EPOCHS):
    train_loss = train_one_epoch(resnet_model, resnet_train_loader)
    valid_metrics = evaluate(resnet_model, resnet_valid_loader)
    scheduler.step(valid_metrics["accuracy"])

    resnet_history.append({"epoch": ep + 1, "train_loss": train_loss, **valid_metrics})

    if valid_metrics["accuracy"] > best_acc:
        best_acc = valid_metrics["accuracy"]
        best_state = {k: v.detach().cpu().clone() for k, v in resnet_model.state_dict().items()}

    print(
        f"ResNet18 Epoch {ep+1:02d} | "
        f"train_loss {train_loss:.4f} | "
        f"valid_loss {valid_metrics['loss']:.4f} | "
        f"acc {valid_metrics['accuracy']:.4f} | "
        f"precision {valid_metrics['precision']:.4f} | "
        f"recall {valid_metrics['recall']:.4f} | "
        f"f1 {valid_metrics['f1']:.4f}"
    )

resnet_model.load_state_dict(best_state)
print("Best ResNet18 valid accuracy:", best_acc)


In [ ]:
resnet_valid_metrics = evaluate(resnet_model, resnet_valid_loader)
print("ResNet18 validation metrics")
print("accuracy :", round(resnet_valid_metrics["accuracy"], 4))
print("precision:", round(resnet_valid_metrics["precision"], 4))
print("recall   :", round(resnet_valid_metrics["recall"], 4))
print("f1       :", round(resnet_valid_metrics["f1"], 4))
print("confusion matrix")
print(resnet_valid_metrics["confusion_matrix"])


In [ ]:
resnet_model.eval()
resnet_preds = []
resnet_probs = []

with torch.no_grad():
    for x in resnet_test_loader:
        x = x.to(device)
        logits = resnet_model(x)
        prob = torch.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(1)

        resnet_probs.extend(prob.cpu().numpy())
        resnet_preds.extend(preds.cpu().numpy())

resnet_submission = pd.read_csv("sample_submission.csv")
resnet_submission["label"] = resnet_preds
resnet_submission.to_csv("resnet18_submit.csv", index=False)

print("저장 완료: resnet18_submit.csv")
print(resnet_submission["label"].value_counts().sort_index())
resnet_submission.head()


## 12. 실험 결과 작성 방향

ResNet18 결과를 SimpleCNN 결과와 비교할 때는 accuracy만 비교하지 말고 confusion matrix에서 FN/FP가 어떻게 바뀌었는지 함께 봅니다. validation 성능이 좋아도 리더보드 점수와 차이가 날 수 있으므로, 단일 split 결과라는 한계를 같이 적는 것이 안전합니다.


## 13. 누락된 실험 기록 복원

이 섹션은 채팅과 `experiment_history.md`에 남아 있던 EXP-06 이후 실험 흔적을 노트북 안에서도 확인하기 위해 추가한 복원 섹션입니다.

학습을 다시 오래 돌리는 코드는 넣지 않고, 이미 저장된 `preds/`, `submissions/`, `models/` 결과 파일을 불러와 비교합니다. 따라서 이 섹션은 빠르게 실행하면서 실험 흐름, 제출 파일, stacking 결과, feature importance, disagreement 샘플을 확인하는 용도입니다.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

ROOT = Path(".")
PREDS = ROOT / "preds"
SUBMISSIONS = ROOT / "submissions"
REPORTS = ROOT / "reports" / "xai"

print("preds files:", len(list(PREDS.glob("*"))))
print("submission files:", len(list(SUBMISSIONS.glob("*.csv"))))
print("reports/xai exists:", REPORTS.exists())

### 13-1. 주요 제출 결과 요약

아래 표는 실험 기록에서 중요한 제출만 추린 것입니다. 현재 최고는 `EXP-26 Ridge without DenseNet`입니다.

In [ ]:
leaderboard_rows = [
    {"exp":"EXP-12", "name":"ConvNeXt-Tiny", "file":"submissions/exp12_convnext_tiny_submit.csv", "valid_acc":0.98084, "lb_score":0.9471153846, "rank":23, "note":"ConvNeXt 단일 최고 기준"},
    {"exp":"EXP-18", "name":"ConvNeXt full fine-tuning", "file":"submissions/exp18_convnext_tiny_full_finetune_submit.csv", "valid_acc":0.97989, "lb_score":0.9471153846, "rank":23, "note":"EXP-12와 동점, 다른 예측"},
    {"exp":"EXP-20", "name":"Ridge stacking", "file":"submissions/exp20_ridge_stacking_alpha100_submit.csv", "valid_acc":0.98755, "lb_score":0.9503205128, "rank":19, "note":"prob+rank stacking, 이전 최고"},
    {"exp":"EXP-21", "name":"Logistic stacking", "file":"submissions/exp21_logistic_stacking_C1_submit.csv", "valid_acc":0.98755, "lb_score":0.9503205128, "rank":19, "note":"Ridge와 동점"},
    {"exp":"EXP-22", "name":"Ridge feature v2", "file":"submissions/exp22_ridge_feature_v2_alpha3000_submit.csv", "valid_acc":0.98276, "lb_score":0.9471153846, "rank":19, "note":"파생 feature 추가 후 하락"},
    {"exp":"EXP-23", "name":"Ridge pruning CNX+DenseNet+ResNet", "file":"submissions/exp23_ridge_pruned_cnx_den_res_alpha100_submit.csv", "valid_acc":0.98755, "lb_score":0.9487179487, "rank":19, "note":"EfficientNet/full ConvNeXt 제거 후 하락"},
    {"exp":"EXP-26", "name":"Ridge without DenseNet", "file":"submissions/exp26_ridge_without_densenet_alpha30_submit.csv", "valid_acc":0.98563, "lb_score":0.9519230769, "rank":19, "note":"현재 최고, test_0244 1->0"},
]
leaderboard_df = pd.DataFrame(leaderboard_rows)
display(leaderboard_df)

### 13-2. 저장된 result JSON 확인

학습을 다시 실행하지 않고, 저장된 result JSON에서 validation metric과 test 예측 분포를 불러옵니다.

In [ ]:
result_files = [
    PREDS / "exp12_convnext_tiny_result.json",
    PREDS / "exp18_convnext_tiny_full_finetune_result.json",
    PREDS / "exp20_ridge_stacking_alpha100_result.json",
    PREDS / "exp21_logistic_stacking_C1_result.json",
    PREDS / "exp26_ridge_without_densenet_alpha30_result.json",
]
rows = []
for path in result_files:
    if not path.exists():
        continue
    data = json.loads(path.read_text())
    metrics = data.get("final_metrics") or data.get("valid_metrics", {})
    rows.append({
        "file": path.name,
        "exp_id": data.get("exp_id"),
        "method/model": data.get("model") or data.get("method"),
        "valid_acc": metrics.get("accuracy"),
        "precision": metrics.get("precision"),
        "recall": metrics.get("recall"),
        "f1": metrics.get("f1"),
        "cm": metrics.get("confusion_matrix"),
        "test_counts": data.get("test_pred_counts"),
        "diff_vs_ridge": data.get("diff_vs_ridge"),
        "diff_vs_exp12": data.get("diff_vs_exp12"),
    })
result_df = pd.DataFrame(rows)
display(result_df)

### 13-3. Ridge stacking feature importance

EXP-24에서 분석한 Ridge coefficient와 permutation importance입니다. 이 분석을 통해 단독 성능이 낮았던 EfficientNet-B0도 stacking에서 보완 정보를 줄 수 있음을 확인했습니다.

![Ridge feature importance](reports/xai/ridge_feature_importance.png)

In [ ]:
coef_path = PREDS / "exp24_ridge_feature_importance_coefficients.csv"
perm_path = PREDS / "exp24_ridge_feature_importance_permutation.csv"
model_summary_path = PREDS / "exp24_ridge_feature_importance_model_summary.csv"

if coef_path.exists():
    print("Coefficient importance")
    display(pd.read_csv(coef_path).sort_values("abs_coef", ascending=False))
if perm_path.exists():
    print("Permutation importance")
    display(pd.read_csv(perm_path).sort_values("acc_importance_mean", ascending=False))
if model_summary_path.exists():
    print("Model-level summary")
    display(pd.read_csv(model_summary_path))

### 13-4. Disagreement 샘플 이미지 Grid

아래 이미지는 stacking 모델들이 다르게 판단한 샘플입니다. 특히 `test_0244.png`는 EXP-20 Ridge에서는 1, 현재 최고 EXP-26에서는 0으로 바뀌었고 leaderboard가 상승했습니다.

![Disagreement samples](reports/xai/disagreement_sample_grid.png)

In [ ]:
from IPython.display import Image, display
img_path = REPORTS / "disagreement_sample_grid.png"
if img_path.exists():
    display(Image(filename=str(img_path)))
else:
    print("이미지 파일이 없습니다:", img_path)

### 13-5. EXP-20, EXP-21, EXP-23, EXP-26 변경 샘플 비교

아래 셀은 각 stacking 후보가 기준 모델과 어떤 샘플에서 달라졌는지 보여줍니다. Grad-CAM/XAI 대상 샘플을 고를 때 이 목록을 우선 사용합니다.

In [ ]:
changed_files = {
    "EXP20 vs EXP12": PREDS / "exp20_ridge_stacking_alpha100_changed_vs_exp12.csv",
    "EXP21 vs Ridge": PREDS / "exp21_logistic_stacking_C1_changed_vs_ridge.csv",
    "EXP23 vs Ridge": PREDS / "exp23_ridge_pruned_cnx_den_res_alpha100_changed_vs_ridge.csv",
    "EXP26 vs Ridge": PREDS / "exp26_ridge_without_densenet_alpha30_changed_vs_ridge.csv",
}
for title, path in changed_files.items():
    print("
", title)
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("missing:", path)

### 13-6. 다음 XAI 작업 메모

다음 단계에서는 이미지 내부 feature importance를 확인합니다.

우선순위:

1. `test_0244.png`: DenseNet 포함 여부에 따라 최종 label이 바뀐 핵심 샘플
2. `test_0267.png`, `test_0371.png`, `test_0535.png`: Ridge/Logistic/pruning에서 판단이 갈린 샘플
3. validation FP/FN 샘플: 정답이 있으므로 Grad-CAM 해석이 더 안전함

주의: Grad-CAM은 “실제 병변을 본다”는 증거가 아니라, 모델이 해당 영역에 더 강하게 반응했다는 시각화입니다.

### 13-7. Grad-CAM 비교

EXP-12 ConvNeXt-Tiny와 EXP-18 full fine-tuning ConvNeXt의 Grad-CAM을 disagreement 샘플에 대해 비교합니다.

주의: Grad-CAM은 모델이 실제 병변을 보았다는 증거가 아니라, 해당 class score를 계산할 때 어느 영역에 더 강하게 반응했는지 보여주는 시각화입니다.

![Grad-CAM comparison](reports/xai/gradcam_convnext_disagreement_comparison.png)

In [ ]:
from IPython.display import Image, display
import pandas as pd
from pathlib import Path

gradcam_img = Path("reports/xai/gradcam_convnext_disagreement_comparison.png")
gradcam_summary = Path("reports/xai/gradcam_convnext_disagreement_summary.csv")

if gradcam_img.exists():
    display(Image(filename=str(gradcam_img)))
else:
    print("Grad-CAM 이미지가 없습니다:", gradcam_img)

if gradcam_summary.exists():
    display(pd.read_csv(gradcam_summary))
else:
    print("Grad-CAM summary가 없습니다:", gradcam_summary)

### 13-8. LIME 비교

EXP-29에서는 EXP-12 ConvNeXt-Tiny를 대상으로 LIME 이미지 설명을 생성했습니다. LIME은 모델 내부 activation이 아니라, 입력 이미지의 superpixel을 가리거나 유지했을 때 예측이 어떻게 바뀌는지를 이용해 설명합니다.

주의: X-ray는 자연 이미지보다 superpixel 경계가 애매할 수 있으므로, LIME 결과는 Grad-CAM과 함께 보조적으로 해석합니다.

![LIME comparison](reports/xai/lime_convnext_disagreement_comparison.png)

In [ ]:
from IPython.display import Image, display
import pandas as pd
from pathlib import Path

lime_img = Path("reports/xai/lime_convnext_disagreement_comparison.png")
lime_summary = Path("reports/xai/lime_convnext_disagreement_summary.csv")

if lime_img.exists():
    display(Image(filename=str(lime_img)))
else:
    print("LIME 이미지가 없습니다:", lime_img)

if lime_summary.exists():
    display(pd.read_csv(lime_summary))
else:
    print("LIME summary가 없습니다:", lime_summary)

### 13-9. XAI 비교 리포트

Grad-CAM, LIME, 모델별 확률, stacking label을 샘플별로 묶은 비교 패널입니다.

![XAI comparison panel](reports/xai/xai_comparison_panel.png)

In [ ]:
from IPython.display import Image, display, Markdown
from pathlib import Path
import pandas as pd

panel = Path("reports/xai/xai_comparison_panel.png")
summary = Path("reports/xai/xai_comparison_summary.csv")
report = Path("reports/xai/xai_comparison_report.md")

if panel.exists():
    display(Image(filename=str(panel)))
if summary.exists():
    display(pd.read_csv(summary))
if report.exists():
    display(Markdown(report.read_text(encoding="utf-8")))

### 13-10. 다중 모델 Grad-CAM 비교

EXP-31에서는 ResNet18, DenseNet121, EfficientNet-B0, ConvNeXt-Tiny의 Grad-CAM을 같은 disagreement 샘플에서 비교합니다.

특히 `test_0244.png`는 EXP-20과 EXP-26의 점수 차이를 만든 핵심 샘플이므로, 각 모델이 어느 영역에 반응했는지 비교합니다.

![Multi-model Grad-CAM](reports/xai/gradcam_multimodel_disagreement_comparison.png)

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import pandas as pd

img = Path("reports/xai/gradcam_multimodel_disagreement_comparison.png")
summary = Path("reports/xai/gradcam_multimodel_disagreement_summary.csv")

if img.exists():
    display(Image(filename=str(img)))
if summary.exists():
    display(pd.read_csv(summary))

### 13-11. Validation FP/FN 분석

EXP-20 Ridge stacking과 현재 최고 제출인 EXP-26 Ridge without DenseNet의 validation 오분류를 비교합니다. test set은 정답을 알 수 없기 때문에, 먼저 정답이 있는 validation set에서 FP/FN 유형을 확인하면 이후 Grad-CAM/LIME 해석 대상을 더 안전하게 고를 수 있습니다.

- EXP-20: FP 4개, FN 9개
- EXP-26: FP 3개, FN 12개
- 해석: EXP-26은 leaderboard 점수는 올랐지만 validation 기준으로는 FP가 줄고 FN이 늘어난 형태입니다. 의료 이미지 분류에서는 FN이 실제 양성을 음성으로 보는 경우라 더 조심해서 확인해야 합니다.


In [ ]:
from IPython.display import Image, display
import pandas as pd
import json

summary_path = "reports/xai/validation_error_summary.csv"
counts_path = "reports/xai/validation_error_counts.json"
image_path = "reports/xai/validation_error_grid.png"

with open(counts_path, "r", encoding="utf-8") as f:
    counts = json.load(f)

print("Validation error counts")
print(counts)

display(Image(filename=image_path))

error_df = pd.read_csv(summary_path)
error_df.head(20)


### 13-12. Validation 판단 변경 샘플 XAI

EXP-33에서는 EXP-20과 EXP-26의 판단이 validation 정답 기준으로 다르게 나온 샘플을 Grad-CAM/LIME으로 비교합니다. 핵심은 모델이 실제 병변을 봤다고 단정하는 것이 아니라, 어떤 모델이 어떤 영역에 상대적으로 더 반응했는지 확인하는 것입니다.

- `train_1229.png`: EXP-20은 FP, EXP-26은 OK
- `train_2325.png`: EXP-20은 OK, EXP-26은 FN
- `train_4170.png`: EXP-20은 OK, EXP-26은 FN

결과적으로 EXP-26은 일부 FP를 줄이는 방향으로 유리했지만, ConvNeXt 계열 판단이 강하게 반영되는 샘플에서는 FN이 늘어날 수 있음을 확인했습니다.


In [ ]:
from IPython.display import Image, display
import pandas as pd

panel_path = "reports/xai/validation_changed_xai_panel.png"
summary_path = "reports/xai/validation_changed_xai_summary.csv"

display(Image(filename=panel_path))
pd.read_csv(summary_path)


### 13-13. EXP-26 Ridge Threshold Search

EXP-34에서는 EXP-26 Ridge decision score의 기준선을 조정했습니다. 기본 기준은 `score >= 0`이면 label 1입니다. threshold를 낮추면 label 1이 늘어나 recall은 올라갈 수 있지만, EXP-26에서 맞춘 정상 샘플을 다시 양성으로 바꿀 위험도 있습니다.

validation 기준 최고는 threshold `-0.20`이지만, 이 경우 `test_0244.png`가 다시 1로 바뀝니다. 이 샘플은 EXP-26에서 0으로 바뀌며 leaderboard가 상승했던 핵심 샘플이라 public score 관점에서는 위험 신호입니다.


In [ ]:
import pandas as pd

threshold_summary = pd.read_csv("preds/exp34_ridge_threshold_summary.csv")
threshold_summary.sort_values(["valid_accuracy", "recall"], ascending=False).head(10)


### 13-14. EXP-35 ConvNeXt 영향 제한

EXP-35에서는 현재 최고인 EXP-26을 기준으로 ConvNeXt 계열 feature의 영향을 줄이는 후보를 비교했습니다. 목적은 validation에서 확인된 FN 증가 가능성을 줄일 수 있는지 확인하는 것입니다.

비교한 방향은 다음과 같습니다.

- ConvNeXt probability를 제거하고 rank만 유지
- ConvNeXt12 또는 ConvNeXt18 full 중 하나 제거
- ResNet18 + EfficientNet-B0만 사용하는 강한 제한 후보

결과적으로 일부 후보는 validation accuracy/recall이 개선됐지만, public score 상승에 중요했던 `test_0244.png`를 다시 label 1로 바꾸는 경향이 있었습니다. 따라서 현재 기준에서는 EXP-26보다 우선 제출할 후보는 낮게 봅니다.


In [ ]:
import pandas as pd

summary = pd.read_csv("preds/exp35_convnext_influence_limit_summary.csv")
summary.head(15)


### 13-15. EXP-40 Swin-Tiny 추가 실험

EXP-40에서는 기존 CNN/ConvNeXt 계열과 다른 Transformer 계열 모델인 Swin-Tiny를 추가했습니다. 첫 stage fine-tuning 설정은 현재 MPS 환경에서 너무 느려 중단했고, 빠른 확인을 위해 pretrained Swin-Tiny의 backbone을 고정하고 classification head만 학습하는 probe 실험으로 진행했습니다.

결과적으로 Swin-Tiny head-only 단독 validation accuracy는 약 0.8994로 낮았습니다. 따라서 단독 제출 후보로 보기 어렵고, stacking 보완 feature로 사용할 수 있는지만 추가 확인했습니다. 하지만 Swin feature를 Ridge stacking에 추가한 후보들도 대부분 `test_0244.png`를 다시 label 1로 바꾸는 방향이라 현재 최고 EXP-26보다 우선 제출할 근거는 부족합니다.


In [ ]:
import json
import pandas as pd

with open("preds/exp40_swin_tiny_result.json", "r", encoding="utf-8") as f:
    swin_result = json.load(f)

print("Swin-Tiny head-only final metrics")
print(swin_result["final_metrics"])
print("test_pred_counts:", swin_result["test_pred_counts"])

stack_summary = pd.read_csv("preds/exp40_swin_stacking_probe_summary.csv")
stack_summary.head(12)


### 13-16. EXP-36 test_0244 고정 stacking 탐색

EXP-36에서는 지금까지 public leaderboard에서 중요하게 보였던 `test_0244.png`를 반드시 label 0으로 유지하는 조건을 걸고 stacking 후보를 탐색했습니다. 목적은 EXP-26의 장점을 유지하면서, 변경 수가 작은 후보 중 validation 지표가 더 좋은 조합을 찾는 것입니다.

가장 우선순위가 높은 후보는 `core_prob_only alpha100`입니다. 이 후보는 ResNet18, EfficientNet-B0, ConvNeXt12, ConvNeXt18 full의 `prob_1`만 사용하고 rank feature는 제외합니다. EXP-26 대비 test 변경은 `test_0025.png` 하나뿐이며, `test_0244.png`는 0으로 유지됩니다.


In [ ]:
import pandas as pd

summary = pd.read_csv("preds/exp36_fixed_0244_stacking_summary.csv")
display(summary.head(15))

changed = pd.read_csv("preds/exp36_fixed0244_core_prob_only_alpha100_changed_vs_exp26.csv")
changed


### 13-17. EXP-37/38 TTA probability와 stacking

EXP-37에서는 기존 강한 모델에 `original + horizontal flip` TTA를 적용해 probability를 다시 생성했습니다. TTA는 학습을 다시 하는 것이 아니라, test/valid 시점에 원본과 좌우반전 이미지를 각각 예측한 뒤 확률을 평균내는 방식입니다.

결과적으로 ConvNeXt12는 validation accuracy가 개선됐지만, ResNet18과 EfficientNet-B0는 소폭 하락했습니다. EXP-38에서는 TTA probability를 Ridge stacking에 넣어 후보를 만들었습니다. 제출 후보는 `tta_all + prob_only` alpha 50/75였고, `test_0244`는 0으로 유지했지만 `test_0025`를 다시 1로 바꾸는 점은 위험 신호입니다.


In [ ]:
import pandas as pd

tta_summary = pd.read_csv("preds/exp37_tta_summary.csv")
display(tta_summary)

stack_summary = pd.read_csv("preds/exp38_tta_stacking_summary.csv")
display(stack_summary.head(12))

changed = pd.read_csv("preds/exp38_tta_all_prob_only_alpha50_changed_vs_exp26.csv")
changed


### 13-18. EXP-39 ConvNeXt seed123 추가 학습

EXP-39에서는 EXP-12와 같은 ConvNeXt-Tiny 마지막 stage+classifier 학습 구조를 유지하고, 학습 seed만 123으로 바꿔 seed diversity를 확보했습니다. 목적은 단독 제출보다는 이후 stacking에 사용할 다른 예측 패턴의 probability를 추가하는 것입니다.

결과 요약:

- best epoch: 4
- validation accuracy: 0.97893
- confusion matrix: `[[263, 5], [17, 759]]`
- EXP-26 대비 test 변경: 19개
- `test_0244`: 0 유지
- `test_0025`: 1로 변경
- `test_0539`: 1로 변경

단독 validation은 EXP-12보다 낮고, 최근 public에서 민감했던 `test_0025`, `test_0539`를 1로 바꾸므로 단독 제출 우선순위는 낮습니다. 다만 stacking diversity 후보로는 저장합니다.


In [ ]:
import pandas as pd
import json

with open("preds/exp39_convnext_tiny_seed123_result.json", "r", encoding="utf-8") as f:
    exp39_result = json.load(f)

print("final metrics")
print(exp39_result["final_metrics"])

summary = pd.read_csv("preds/exp39_convnext_tiny_seed123_summary.csv")
changed = pd.read_csv("preds/exp39_convnext_tiny_seed123_changed_vs_exp26.csv")

display(summary)
display(changed)


### 13-19. EXP-41 5-fold OOF 설계

EXP-41에서는 단일 validation stacking의 과적합 위험을 줄이기 위해 5-fold OOF 설계를 고정했습니다. OOF는 각 train 샘플을 그 샘플을 학습에 사용하지 않은 fold 모델로 예측해 meta-model 학습용 feature를 만드는 방식입니다.

설정:

- `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`
- fold별 label 비율 유지
- OOF prediction 포맷 고정
- 이후 Ridge/Logistic/TabM이 같은 OOF feature table을 사용하도록 설계

TabM은 단일 validation에서는 과적합 위험이 크므로, OOF feature가 만들어진 뒤 비교하는 것이 적절합니다.


In [ ]:
import pandas as pd

folds = pd.read_csv("oof/folds_seed42.csv")
fold_summary = pd.read_csv("oof/folds_seed42_summary.csv")

display(fold_summary)
print("rows:", len(folds))
print("unique files:", folds["file_name"].nunique())
print("fold counts")
print(folds["fold"].value_counts().sort_index())


### 13-20. EXP-42 EfficientNetV2-S 단일 split

EfficientNetV2-S를 새 CNN 계열 후보로 확인했습니다. 첫 시도는 계산량을 줄이기 위해 `features[7] + classifier`만 학습했습니다.

결과적으로 validation accuracy는 0.87261로 낮았고, confusion matrix는 `[[257, 11], [122, 654]]`였습니다. 특히 FN이 크게 증가했기 때문에 의료 이미지 분류 관점에서는 제출 후보로 적합하지 않습니다.

민감 샘플도 `test_0244=1`, `test_0025=1`로 바뀌어 기존 public 결과 기준 위험 신호가 있습니다. 따라서 현재 설정의 EfficientNetV2-S는 단독 제출 및 stacking feature 후보에서 제외하고, 재시도한다면 더 넓은 fine-tuning 범위로 다시 확인합니다.



In [ ]:
import json
import pandas as pd

with open("preds/exp42_efficientnet_v2_s_result.json", "r", encoding="utf-8") as f:
    exp42_result = json.load(f)

print("EXP-42 final metrics")
print(exp42_result["final_metrics"])

print("\nEXP-42 summary")
display(pd.read_csv("preds/exp42_efficientnet_v2_s_summary.csv"))

print("\nEXP-26 대비 변경 샘플 일부")
display(pd.read_csv("preds/exp42_efficientnet_v2_s_changed_vs_exp26.csv").head(30))



### 13-21. EXP-43 RegNetY-3.2GF 단일 split

RegNetY 계열을 새 CNN diversity 후보로 확인했습니다. torchvision에는 RegNetY-4GF weight가 없어 가장 가까운 `RegNetY-3.2GF`를 사용했고, `trunk_output.block4 + fc`만 학습했습니다.

결과는 validation accuracy 0.95211, confusion matrix `[[261, 7], [43, 733]]`입니다. EXP-42 EfficientNetV2-S보다 FN이 크게 줄었지만, 기존 ConvNeXt/Ridge 후보보다 성능이 낮아 단독 제출보다는 stacking 후보로 보는 것이 맞습니다.

민감 샘플 `test_0244`, `test_0025`, `test_0539`는 모두 0으로 유지했습니다. 이 점은 이후 stacking에 넣어볼 가치가 있는 긍정 신호입니다.



In [ ]:
import json
import pandas as pd

with open("preds/exp43_regnet_y_3_2gf_result.json", "r", encoding="utf-8") as f:
    exp43_result = json.load(f)

print("EXP-43 final metrics")
print(exp43_result["final_metrics"])

print("\nEXP-43 summary")
display(pd.read_csv("preds/exp43_regnet_y_3_2gf_summary.csv"))

print("\nEXP-26 대비 변경 샘플 일부")
display(pd.read_csv("preds/exp43_regnet_y_3_2gf_changed_vs_exp26.csv").head(30))



### 13-22. EXP-44 RegNetY 포함 Ridge stacking

EXP-43 RegNetY-3.2GF는 단독 성능은 낮았지만 민감 샘플을 모두 0으로 유지했습니다. 그래서 EXP-44에서는 RegNetY를 기존 stacking feature에 추가해 보완 정보로 작동하는지 확인했습니다.

가장 좋은 후보는 `drop_resnet18_add_reg alpha=150`입니다. 이 후보는 ResNet18을 제외하고 EfficientNet-B0, ConvNeXt12, ConvNeXt18 full, RegNetY-3.2GF의 `prob + rank` feature를 사용했습니다.

Validation accuracy는 0.98659이고, confusion matrix는 `[[265, 3], [11, 765]]`입니다. EXP-26 대비 test 예측은 5개만 바뀌었고, `test_0244`, `test_0025`, `test_0539`는 모두 0으로 유지했습니다.



In [ ]:
import pandas as pd

exp44_summary = pd.read_csv("preds/exp44_regnet_stacking_summary.csv")
cols = [
    "spec", "alpha", "n_features", "valid_accuracy", "valid_precision",
    "valid_recall", "valid_f1", "tn", "fp", "fn", "tp",
    "changed_vs_exp26", "test_0244_pred", "test_0025_pred",
    "test_0539_pred", "candidate"
]
print("EXP-44 top candidates")
display(exp44_summary[cols].head(12))

print("\nEXP-44 selected candidate changes vs EXP-26")
display(pd.read_csv("preds/exp44_regnet_drop_resnet18_add_reg_alpha150_changed_vs_exp26.csv"))



### 13-23. EXP-45 ConvNeXt OOF fold0 smoke test

EXP-45에서는 5-fold OOF 전체 학습 전에 ConvNeXt-Tiny fold0만 먼저 학습해 pipeline을 검증했습니다. 목적은 leaderboard 제출이 아니라 OOF probability 저장 형식, row 수, 중복 여부, 학습 시간을 확인하는 것입니다.

fold0 결과는 validation accuracy 0.98467, confusion matrix `[[266, 3], [13, 762]]`입니다. OOF probability는 1,044 rows, test probability는 624 rows로 저장되었고 file_name 중복은 없었습니다. fold0 학습 시간은 433.2초로, 5-fold 전체는 약 36분 정도로 예상됩니다.



In [ ]:
import json
import pandas as pd

summary = pd.read_csv("oof/preds/exp45_convnext_tiny_fold0_summary.csv")
oof_probs = pd.read_csv("oof/preds/exp45_convnext_tiny_fold0_oof_probs.csv")
test_probs = pd.read_csv("oof/preds/exp45_convnext_tiny_fold0_test_probs.csv")
sub = pd.read_csv("oof/submissions/exp45_convnext_tiny_fold0_submit.csv")

print("EXP-45 summary")
display(summary)

print("OOF format check")
print("oof shape:", oof_probs.shape)
print("oof duplicated file_name:", oof_probs["file_name"].duplicated().sum())
print("oof fold counts:", oof_probs["fold"].value_counts().to_dict())
print("test shape:", test_probs.shape)
print("test duplicated file_name:", test_probs["file_name"].duplicated().sum())
print("test fold counts:", test_probs["fold"].value_counts().to_dict())

print("sensitive samples")
for file_name in ["test_0244.png", "test_0025.png", "test_0539.png"]:
    row = test_probs.loc[test_probs["file_name"] == file_name].iloc[0]
    print(file_name, "pred=", int(row["pred_label"]), "prob_1=", round(float(row["prob_1"]), 5))



### 13-24. EXP-46 ConvNeXt 5-fold OOF 완성

EXP-46에서는 fold1~4를 추가 학습해 EXP-45의 fold0 결과와 합쳐 ConvNeXt-Tiny 5-fold OOF probability를 완성했습니다.

OOF rows는 전체 train 샘플 수와 같은 5,216개이고, 중복 file_name은 없습니다. Test probability는 fold별 3,120 rows와 fold 평균 624 rows로 저장했습니다.

OOF accuracy는 0.97795, confusion matrix는 `[[1312, 29], [86, 3789]]`입니다. 이 값은 단일 validation 점수보다 보수적인 성능 추정이며, 이후 OOF 기반 Ridge/Logistic meta-model의 입력 feature로 사용합니다.



In [ ]:
import json
import pandas as pd

with open("oof/preds/exp46_convnext_tiny_combined_result.json", "r", encoding="utf-8") as f:
    exp46 = json.load(f)

print("EXP-46 combined result")
print(json.dumps(exp46, indent=2, ensure_ascii=False))

fold_summary = pd.read_csv("oof/preds/exp46_convnext_tiny_fold_summary.csv")
oof_probs = pd.read_csv("oof/preds/exp46_convnext_tiny_oof_probs.csv")
test_by_fold = pd.read_csv("oof/preds/exp46_convnext_tiny_test_probs_by_fold.csv")
test_mean = pd.read_csv("oof/preds/exp46_convnext_tiny_test_probs_mean.csv")

print("\nfold summary")
display(fold_summary)

print("OOF rows:", len(oof_probs), "duplicates:", oof_probs["file_name"].duplicated().sum())
print("OOF fold counts:", oof_probs["fold"].value_counts().sort_index().to_dict())
print("test by fold rows:", len(test_by_fold))
print("test mean rows:", len(test_mean))

print("\nsensitive samples from test mean")
display(test_mean[test_mean["file_name"].isin(["test_0244.png", "test_0025.png", "test_0539.png"])])



### 13-25. EXP-47 OOF Ridge baseline

EXP-47에서는 ConvNeXt 5-fold OOF probability 하나만 사용해 RidgeClassifier baseline을 만들었습니다. 아직 여러 모델을 합친 stacking은 아니며, OOF meta-model 파이프라인이 정상 작동하는지 확인하는 단계입니다.

최고 OOF 후보는 `prob1_only alpha=1000`으로 OOF accuracy 0.98025를 기록했습니다. 그러나 이 후보는 `test_0244`, `test_0025`를 모두 1로 바꾸므로 제출 우선순위는 낮습니다.

민감 샘플을 유지하는 가장 나은 후보는 `prob1_margin alpha=1000`이며 OOF accuracy는 0.97872입니다. 이 실험의 결론은 OOF Ridge 구조는 작동하지만, 실제 성능 개선을 위해서는 ResNet18 등 다른 모델의 OOF feature 추가가 필요하다는 것입니다.



In [ ]:
import pandas as pd

summary = pd.read_csv("oof/preds/exp47_oof_ridge_baseline_summary.csv")
cols = [
    "spec", "alpha", "oof_accuracy", "oof_precision", "oof_recall", "oof_f1",
    "tn", "fp", "fn", "tp", "test_changed_vs_direct", "test_label0", "test_label1",
    "test_0244_pred", "test_0025_pred", "test_0539_pred"
]

print("EXP-47 top OOF Ridge candidates")
display(summary[cols].head(12))

print("\nDirect ConvNeXt OOF mean")
display(summary.loc[summary["spec"] == "direct_convnext_oof_mean", cols])

print("\nSensitive-safe candidates")
safe = summary[(summary["test_0244_pred"] == 0) & (summary["test_0025_pred"] == 0)]
display(safe[cols].head(10))



### 13-21. EXP-48~51 OOF multi-model 확장

EXP-48~51에서는 단일 validation 기반 stacking에서 벗어나 5-fold OOF 기반 multi-model stacking을 확장했습니다.

핵심 흐름:

- EXP-48: ResNet18 5-fold OOF 생성, OOF accuracy 0.98524
- EXP-49: ConvNeXt+ResNet OOF stacking, meta-CV 최고 0.98735였지만 민감 샘플 유지 실패
- EXP-50: EfficientNet-B0 5-fold OOF 생성, OOF accuracy 0.98505, `test_0244/test_0025/test_0539` 모두 0 유지
- EXP-51: ConvNeXt+EfficientNet 중심 OOF stacking 후보 생성

제출 후보 관점:

- 보수적 후보: `oof/submissions/exp51_logistic_cnx_eff_prob_rank_0p3_submit.csv`
- 공격적 후보: `oof/submissions/exp51_ridge_cnx_eff_prob_only_50_submit.csv`

해석상 중요한 점은 OOF/meta-CV 점수만으로 제출을 결정하지 않고, EXP-26 대비 변경 샘플 수와 민감 샘플 예측을 함께 확인한다는 것입니다.


In [ ]:
import pandas as pd

exp51_summary = pd.read_csv("oof/preds/exp51_oof_3model_stacking_summary.csv")
cols = [
    "spec", "model", "param", "meta_cv_accuracy", "meta_cv_precision", "meta_cv_recall",
    "meta_cv_f1", "meta_cv_fp", "meta_cv_fn", "test_label0", "test_label1",
    "changed_vs_exp26", "test_0244_pred", "test_0025_pred", "test_0539_pred", "candidate"
]
display(exp51_summary[cols].head(15))


#### EXP-51 결과 해석

상위 후보는 대부분 ResNet18을 제외한 ConvNeXt+EfficientNet-B0 조합에서 나왔습니다. ResNet18은 OOF 성능은 높았지만 test 민감 샘플을 1로 밀어 올리는 경향이 있었고, EfficientNet-B0는 해당 샘플들을 0으로 유지했습니다. 따라서 현재 제출 후보는 `meta_cv_accuracy` 최고 후보와 EXP-26 대비 변경 수가 적은 후보를 나누어 판단합니다.

- `exp51_ridge_cnx_eff_prob_only_50_submit.csv`: meta-CV accuracy 0.98677, EXP-26 대비 8개 변경, 민감 샘플 3개 유지
- `exp51_logistic_cnx_eff_prob_rank_0p3_submit.csv`: meta-CV accuracy 0.98639, EXP-26 대비 4개 변경, 민감 샘플 3개 유지

의료 이미지 분류 해석 관점에서는 FN/FP 균형도 함께 확인해야 하며, leaderboard에서는 1~2개 샘플 차이로 순위가 바뀔 수 있으므로 과도하게 단정하지 않습니다.


### 13-22. EXP-52 low-change search

EXP-52에서는 EXP-51 최고 후보 주변에서 새 학습 없이 Ridge/Logistic regularization과 threshold만 조정했습니다. 목표는 현재 최고 후보 대비 변경 수를 1~4개 수준으로 제한하여 public leaderboard에서 추가 1샘플 개선 가능성을 확인하는 것입니다.

1순위 후보:

- `oof/submissions/exp52_logistic_cnx_eff_prob_only_p0p25_t0p0_submit.csv`
- EXP-51 최고 대비 변경 1개: `test_0490` 1->0
- 민감 샘플 `test_0244`, `test_0025`, `test_0539`는 모두 0 유지

주의할 점은 기대 상승폭이 작다는 것입니다. 변경 샘플이 1개뿐이므로 맞으면 +1샘플, 틀리면 -1샘플 수준의 실험입니다.


In [ ]:
import pandas as pd

exp52_summary = pd.read_csv("oof/preds/exp52_low_change_summary.csv")
cols = [
    "spec", "model", "param", "threshold", "meta_cv_accuracy", "meta_cv_recall",
    "meta_cv_fp", "meta_cv_fn", "test_label0", "test_label1",
    "changed_vs_exp26", "changed_vs_exp51_best",
    "test_0244_pred", "test_0025_pred", "test_0539_pred", "candidate"
]
display(exp52_summary[cols].head(15))


### 13-23. EXP-52 제출 결과와 EXP-53 constraint search

EXP-52 1순위 후보 `exp52_logistic_cnx_eff_prob_only_p0p25_t0p0_submit.csv`는 leaderboard score `0.9567307692`를 기록해 EXP-51 대비 정확히 1샘플 개선되었습니다. 변경된 샘플은 `test_0490: 1->0` 하나였으므로, public 기준에서는 이 방향이 맞았을 가능성이 높습니다.

반대로 EXP-52 2순위 후보 `exp52_ridge_cnx_eff_prob_margin_p500_tm0p025_submit.csv`는 `test_0201`까지 1->0으로 바꾸면서 `0.9551282051`로 하락했습니다. 따라서 이후 low-change search에서는 `test_0490=0`, `test_0201=1`을 보호 조건으로 둡니다.

EXP-53에서는 이 보호 조건을 적용해 EXP-52 winner 대비 1~3개만 바꾸는 후보를 탐색했지만, 최종 candidate는 0개였습니다. 현재 ConvNeXt+EfficientNet OOF 조합 안에서는 더 미세한 제출 후보를 만들기보다 새 seed OOF 정보원을 확보하는 쪽이 타당합니다.


In [ ]:
import pandas as pd

exp53_summary = pd.read_csv("oof/preds/exp53_constraint_low_change_summary.csv")
cols = [
    "spec", "model", "param", "threshold", "meta_cv_accuracy", "meta_cv_recall",
    "test_label0", "test_label1", "changed_vs_current_best",
    "test_0025_pred", "test_0539_pred", "test_0244_pred", "test_0490_pred", "test_0201_pred",
    "passes_protected", "passes_label_band", "passes_low_change_current", "candidate"
]

print("EXP-53 candidate count:", int(exp53_summary["candidate"].sum()))
display(exp53_summary[cols].head(15))


### 13-24. EXP-54/55 ConvNeXt seed OOF와 stacking

EXP-54에서는 ConvNeXt-Tiny를 같은 구조로 seed만 123으로 바꿔 5-fold OOF를 추가 생성했습니다. OOF accuracy는 `0.98102`였고, 단독 mean 제출은 `test_0539=1`, `test_0490=1`이라 제출 우선순위가 낮습니다. 특히 `test_0490=0`은 EXP-52에서 실제 public 개선과 연결되었으므로 단독 제출은 보류합니다.

EXP-55에서는 EXP-54 seed123 OOF를 기존 ConvNeXt/EfficientNet-B0 OOF와 결합해 Ridge/Logistic stacking을 다시 탐색했습니다. 가장 좋은 제출 후보는 `exp55_ridge_cnx123_eff_prob_rank_p50_tm0p1_submit.csv`이며, 현재 최고 EXP-52 대비 `test_0222` 하나만 `1->0`으로 바꿉니다. 보호 샘플 5개는 모두 유지합니다.


In [ ]:
import json
from pathlib import Path
import pandas as pd

exp54 = json.loads(Path("oof/preds/exp54_convnext_tiny_seed123_combined_result.json").read_text())
print("EXP-54 OOF metrics")
print(json.dumps(exp54["oof_metrics"], indent=2, ensure_ascii=False))
print("EXP-54 watched samples")
print(json.dumps(exp54["watched_samples"], indent=2, ensure_ascii=False))

exp55_summary = pd.read_csv("oof/preds/exp55_convnext_seed_stacking_summary.csv")
cols = [
    "spec", "model", "param", "threshold", "meta_cv_accuracy", "meta_cv_recall",
    "meta_cv_fp", "meta_cv_fn", "test_label0", "test_label1",
    "changed_vs_current_best", "test_0025_pred", "test_0539_pred", "test_0244_pred",
    "test_0490_pred", "test_0201_pred", "candidate"
]
print("EXP-55 candidate count:", int(exp55_summary["candidate"].sum()))
display(exp55_summary[cols].head(12))


#### EXP-55 제출 결과 업데이트

`exp55_ridge_cnx123_eff_prob_rank_p50_tm0p1_submit.csv` 제출 결과 leaderboard score는 `0.9551282051`이었습니다. 현재 최고 EXP-52의 `0.9567307692`보다 낮으며, 이 후보는 현재 최고 대비 `test_0222` 하나만 `1->0`으로 바꿨습니다.

따라서 public 기준에서는 `test_0222=1`을 유지하는 편이 유리했을 가능성이 높습니다. 이후 constrained search에서는 기존 보호 샘플에 `test_0222=1`을 추가합니다. EXP-55 2순위 후보는 `test_0504` 하나만 바꾸지만, 바로 제출하기보다 새 보호 조건을 적용한 후 다시 판단합니다.


### 13-25. EXP-56 보호 샘플 기반 low-change search

EXP-55 제출 결과 `test_0222: 1->0` 변경이 leaderboard 하락과 연결되었으므로, EXP-56에서는 `test_0222=1`을 보호 조건에 추가했습니다. 기존 보호 샘플까지 포함하면 `test_0025=0`, `test_0539=0`, `test_0244=0`, `test_0490=0`, `test_0201=1`, `test_0222=1`을 유지합니다.

최종 후보는 `exp56_ridge_cnx_eff_compact_p3000_t0p025_submit.csv` 하나입니다. 현재 최고 EXP-52 대비 `test_0504` 하나만 `1->0`으로 바꾸며, 보호 샘플 6개는 모두 유지합니다.


In [ ]:
import pandas as pd

exp56_summary = pd.read_csv("oof/preds/exp56_protected_low_change_summary.csv")
cols = [
    "spec", "model", "param", "threshold", "meta_cv_accuracy", "meta_cv_recall",
    "meta_cv_fp", "meta_cv_fn", "test_label0", "test_label1", "changed_vs_current_best",
    "test_0025_pred", "test_0539_pred", "test_0244_pred", "test_0490_pred",
    "test_0201_pred", "test_0222_pred", "candidate"
]
print("EXP-56 candidate count:", int(exp56_summary["candidate"].sum()))
display(exp56_summary[cols].head(12))


#### EXP-56 제출 결과 업데이트

`exp56_ridge_cnx_eff_compact_p3000_t0p025_submit.csv` 제출 결과 leaderboard score는 `0.9583333333`이었습니다. 이전 최고 EXP-52의 `0.9567307692`보다 정확히 1샘플 개선되었습니다.

이 후보는 현재 최고 대비 `test_0504` 하나만 `1->0`으로 바꿨으므로, public 기준에서는 `test_0504=0` 방향이 맞았을 가능성이 높습니다. 이후 constrained search에서는 기존 보호 샘플에 `test_0504=0`을 추가합니다.


### 13-26. EXP-57 EXP-56 기준 보호 샘플 low-change search

EXP-56이 새 최고점이 되었으므로 `test_0504=0`까지 보호 조건에 추가하고, 같은 OOF feature 공간에서 추가 low-change 후보가 있는지 확인했습니다. 보호 샘플은 `test_0025=0`, `test_0539=0`, `test_0244=0`, `test_0490=0`, `test_0201=1`, `test_0222=1`, `test_0504=0`입니다.

결과적으로 meta-CV가 높은 조합은 있었지만 보호 샘플 7개를 모두 유지하는 비중복 제출 후보는 0개였습니다. 따라서 현재 ConvNeXt/ConvNeXt seed123/EfficientNet-B0 OOF feature 공간에서는 추가 제출 후보를 억지로 만들기보다 calibration 확인 또는 새 OOF 정보원 추가가 더 타당합니다.



In [ ]:
import pandas as pd

exp57_summary = pd.read_csv("oof/preds/exp57_protected_low_change_summary.csv")
print("EXP-57 rows:", len(exp57_summary))
print("candidate count:", int(exp57_summary["candidate"].sum()))
print("protected pass count:", int(exp57_summary["passes_protected"].sum()))
print("best meta-CV:", round(exp57_summary["meta_cv_accuracy"].max(), 5))

cols = [
    "spec", "model", "param", "threshold", "meta_cv_accuracy",
    "changed_vs_current_best", "test_label0", "test_label1",
    "test_0025_pred", "test_0539_pred", "test_0244_pred",
    "test_0490_pred", "test_0201_pred", "test_0222_pred", "test_0504_pred",
    "passes_protected", "passes_low_change_current", "candidate",
]
display(exp57_summary[cols].head(10))



### 13-27. EXP-58 Calibrated OOF stacking

EXP-57에서 보호 샘플 7개를 모두 유지하는 추가 low-change 후보가 없었기 때문에, EXP-58에서는 Ridge/Logistic/LinearSVC meta-model에 sigmoid/isotonic calibration을 적용했습니다. 목적은 같은 OOF feature라도 score를 확률 형태로 보정했을 때 decision boundary 주변 샘플이 다르게 정렬되는지 확인하는 것입니다.

결과적으로 후보는 1개 생성되었습니다. `exp58_svc_all_margin_stats_isotonic_p0p2_t0p575_submit.csv`는 보호 샘플 7개를 모두 유지하면서 EXP-56 현재 최고 대비 `test_0188`, `test_0568` 두 샘플만 `0->1`로 바꿉니다. 두 샘플이 모두 맞으면 public 기준 +2샘플 개선 가능성이 있지만, 둘 다 틀리면 -2샘플 하락할 수 있어 중간 리스크 후보로 봅니다.



In [ ]:
import pandas as pd

exp58_summary = pd.read_csv("oof/preds/exp58_calibrated_stacking_summary.csv")
print("EXP-58 rows:", len(exp58_summary))
print("candidate count:", int(exp58_summary["candidate"].sum()))

cols = [
    "spec", "model", "param", "calibration", "threshold",
    "meta_cv_accuracy", "meta_cv_recall", "test_label0", "test_label1",
    "changed_vs_current_best", "passes_protected", "candidate",
]
display(exp58_summary[exp58_summary["candidate"]][cols])

changed = pd.read_csv("oof/preds/exp58_svc_all_margin_stats_isotonic_p0p2_t0p575_changed_vs_current_best.csv")
display(changed)



#### EXP-58 제출 결과 업데이트

`exp58_svc_all_margin_stats_isotonic_p0p2_t0p575_submit.csv` 제출 결과 leaderboard score는 `0.9551282051`이었습니다. 현재 최고 EXP-56의 `0.9583333333`보다 낮으므로, EXP-56 대비 `test_0188`, `test_0568` 두 샘플을 모두 `0->1`로 바꾸는 방향은 public 기준에서 불리했습니다.

따라서 현재 최고 후보는 계속 `exp56_ridge_cnx_eff_compact_p3000_t0p025_submit.csv`입니다. EXP-58 결과는 calibration 자체가 항상 나쁘다는 뜻은 아니지만, 현재 조합에서는 recall을 더 올리는 방향이 FP 증가로 이어졌을 가능성이 큽니다.



#### 리더보드 순위 업데이트

최신 public leaderboard 기준 현재 최고 제출 `exp56_ridge_cnx_eff_compact_p3000_t0p025_submit.csv`의 순위는 13위로 확인되었습니다. 점수는 `0.9583333333`이며, EXP-58 제출 이후에도 선택된 최고 제출은 EXP-56입니다.



#### EXP-52 순위 정정

`exp52_logistic_cnx_eff_prob_only_p0p25_t0p0_submit.csv`는 당시 public leaderboard 기준 14위였습니다. 이후 EXP-56이 `0.9583333333`으로 갱신되며 현재 최고 순위는 13위입니다.



### 13-28. EXP-65 ConvNeXt V2 Tiny 사전 검토

ConvNeXt V2 Tiny를 새 OOF 정보원 후보로 검토했습니다. 현재 환경에 `timm==1.0.27`을 추가했고, `convnextv2_tiny.fcmae_ft_in22k_in1k` pretrained weight 로딩과 224 입력 forward smoke test가 성공했습니다. 모델 파라미터는 약 27.87M이며, 1차 fine-tuning 후보인 `stage3 + head`는 약 15.49M 파라미터를 학습합니다.

기존 ConvNeXt OOF 기록상 5-fold 학습은 약 34~37분 수준이었고, ConvNeXt V2 Tiny는 비슷하거나 약간 더 걸릴 것으로 예상됩니다. 따라서 `stage3 + head`, 224 입력, 6 epochs 기준 5-fold OOF는 약 40~70분으로 추정되며 10시간 초과 장기 실험 게이트 대상은 아닙니다.

목적은 단독 제출이 아니라 ConvNeXt V1/EfficientNet/ResNet과 다른 오류 방향을 가진 OOF feature를 확보하는 것입니다. 다음 단계는 `scripts/run_exp65_convnextv2_tiny_oof.py` 작성 후 fold0 smoke test입니다. 상세 검토는 `oof/exp65_convnextv2_tiny_precheck.md`에 정리했습니다.



### 13-29. EXP-65 ConvNeXt V2 Tiny 5-fold OOF 결과

ConvNeXt V2 Tiny `convnextv2_tiny.fcmae_ft_in22k_in1k`를 `stage3 + head`만 fine-tuning하는 방식으로 5-fold OOF를 생성했습니다. 전체 OOF accuracy는 `0.97680`, recall은 `0.97961`, confusion matrix는 `[[1299, 42], [79, 3796]]`입니다. 총 학습 시간은 약 95.3분이었습니다.

단독 test mean 제출은 보류합니다. test mean label 분포가 `0/1 = 208/416`으로 공격적이고, EXP-56 현재 최고 대비 30개 샘플이 바뀝니다. 특히 `test_0025`, `test_0539`, `test_0244`, `test_0490`, `test_0504`를 모두 1로 예측해 기존 public 결과 기반 보호 방향을 많이 깹니다.

다만 ConvNeXt V2는 기존 모델과 오류 방향이 다를 가능성이 있으므로, EXP-66에서 stacking feature로 제한적으로 추가해볼 가치는 있습니다. 이때 단독 성능보다 보호 샘플 유지 여부와 EXP-56 대비 변경 수를 우선 확인합니다.



In [ ]:
import json
from pathlib import Path
import pandas as pd

exp65 = json.loads(Path("oof/preds/exp65_convnextv2_tiny_stage3_combined_result.json").read_text())
print(json.dumps(exp65["oof_metrics"], indent=2))
print("test_mean_pred_counts:", exp65["test_mean_pred_counts"])
print("watched_samples:")
print(json.dumps(exp65["watched_samples"], indent=2))

fold_summary = pd.read_csv("oof/preds/exp65_convnextv2_tiny_stage3_fold_summary.csv")
display(fold_summary[["fold", "best_epoch", "valid_accuracy", "valid_recall", "valid_cm", "elapsed_sec", "test_label0", "test_label1"]])

changed = pd.read_csv("oof/preds/exp65_convnextv2_tiny_stage3_changed_vs_exp56.csv")
print("changed_vs_exp56:", len(changed))
display(changed.head(30))



## 13-31. EXP-66 ConvNeXt V2 포함 OOF stacking

EXP-65 ConvNeXt V2 Tiny 단독 mean은 보호 샘플을 많이 1로 바꿔 제출 후보로 부적절했다. 이번에는 단독 제출이 아니라 기존 ConvNeXt / ConvNeXt seed123 / EfficientNet-B0 OOF feature에 ConvNeXt V2 probability를 제한적으로 추가해 Ridge/Logistic stacking 후보를 탐색했다.

핵심 결과:

- 1순위 후보: `oof/submissions/exp66_ridge_all_prob_stats_with_cnxv2_p10_t0p15_submit.csv`
- meta-CV accuracy: `0.98696`
- confusion matrix: `[[1317, 24], [44, 3831]]`
- test label 0/1: `229/395`
- EXP-56 대비 변경: `test_0141.png` 하나만 `1 -> 0`
- 보호 샘플 7개 유지: `test_0025=0`, `test_0539=0`, `test_0244=0`, `test_0490=0`, `test_0201=1`, `test_0222=1`, `test_0504=0`
- EXP-58에서 하락했던 `test_0188`, `test_0568`은 둘 다 `0` 유지

해석: ConvNeXt V2는 단독으로는 과하게 label 1을 예측했지만, stacking feature로 제한적으로 사용할 때는 보호 샘플을 깨지 않는 1개 변경 후보를 만들었다. 다만 새로 바뀐 `test_0141`의 실제 정답은 알 수 없으므로, 제출 후보로는 합리적이지만 기대 개선 폭은 최대 1샘플 수준이다.


In [ ]:
import pandas as pd

summary_path = 'oof/preds/exp66_convnextv2_stacking_summary.csv'
changed_path = 'oof/preds/exp66_ridge_all_prob_stats_with_cnxv2_p10_t0p15_changed_vs_current_best.csv'
submit_path = 'oof/submissions/exp66_ridge_all_prob_stats_with_cnxv2_p10_t0p15_submit.csv'

summary = pd.read_csv(summary_path)
changed = pd.read_csv(changed_path)
submit = pd.read_csv(submit_path)

print('EXP-66 candidate summary')
display(summary[summary['candidate'] == True].head(6))

print('Changed vs EXP-56 current best')
display(changed)

print('Submission shape and label distribution')
print(submit.shape)
print(submit['label'].value_counts().sort_index())

watch = ['test_0025.png', 'test_0539.png', 'test_0244.png', 'test_0490.png',
         'test_0201.png', 'test_0222.png', 'test_0504.png', 'test_0188.png', 'test_0568.png']
display(submit[submit['file_name'].isin(watch)])


### EXP-66 leaderboard 결과 반영

`oof/submissions/exp66_ridge_all_prob_stats_with_cnxv2_p10_t0p15_submit.csv` 제출 결과 public score는 `0.9567307692`였다. 이는 현재 최고인 EXP-56 `0.9583333333`보다 낮다.

EXP-66은 EXP-56 대비 `test_0141.png` 하나만 `1 -> 0`으로 바꾼 후보였으므로, public 기준에서는 `test_0141=0` 방향이 이득이 아니었던 것으로 해석한다. 따라서 다음 low-change 탐색에서는 `test_0141=1`을 우선 유지하고, 최고 기준은 계속 EXP-56으로 둔다.


## 13-32. EXP-67/68 RegNetY OOF와 stacking

RegNetY-3.2GF를 5-fold OOF 모델로 추가했다. 목적은 단독 최고 모델을 만드는 것이 아니라 ConvNeXt/EfficientNet과 다른 CNN family의 오류 방향을 확보하는 것이다.

EXP-67 RegNetY OOF 결과:

- OOF accuracy: `0.96166`
- precision: `0.99039`
- recall: `0.95768`
- F1: `0.97376`
- confusion matrix: `[[1305, 36], [164, 3711]]`
- test mean label 0/1: `273/351`
- EXP-56 대비 변경 수: `69`

해석: precision은 높지만 recall이 낮고 FN이 164개로 많다. 따라서 단독 제출은 부적절하다. 다만 `test_0025`, `test_0539`, `test_0188`, `test_0568`을 0으로 강하게 보는 특성이 있어 보조 feature로는 확인 가치가 있었다.

EXP-68 RegNetY 포함 stacking 결과:

- 최고 meta-CV accuracy: `0.98735`
- 보호 샘플 8개 유지 candidate: `0`
- 저장된 제출 파일: 없음

해석: RegNetY feature는 일부 meta-CV를 올렸지만, 상위 후보들이 `test_0539`, `test_0490`, `test_0504`, `test_0141` 등 보호 방향을 자주 깨서 제출 후보로 남지 않았다. 현재 기준에서는 RegNetY를 바로 제출 후보에 쓰기보다 이후 더 큰 OOF pool에서 선택적으로 쓰는 편이 낫다.


In [ ]:
import json
import pandas as pd

with open('oof/preds/exp67_regnet_y_3_2gf_combined_result.json') as f:
    exp67 = json.load(f)
print('EXP-67 OOF metrics')
print(exp67['oof_metrics'])
print('EXP-67 watched samples')
display(pd.DataFrame(exp67['watched_samples']).T)

exp68 = pd.read_csv('oof/preds/exp68_regnet_stacking_summary.csv')
print('EXP-68 candidate count:', int(exp68['candidate'].sum()))
display(exp68.head(10))


### 13-43. EXP-69 DINOv2 feature extractor + Ridge/Logistic

DINOv2는 ConvNeXt/EfficientNet/RegNet과 다른 ViT 기반 foundation feature 계열입니다. 이번 실험은 DINOv2를 직접 fine-tuning하지 않고, `vit_small_patch14_dinov2`를 frozen feature extractor로 사용해 384차원 feature를 추출한 뒤 Ridge/Logistic meta-model로 분류 가능성을 확인했습니다.

핵심 결과는 다음과 같습니다.

- train/test feature rows: 5,216 / 624
- feature dimension: 384
- feature extraction time: 약 623.6초
- 최고 meta-CV accuracy: 0.97373
- 보호 샘플 유지 candidate: 0개
- 제출 파일: 없음

해석: DINOv2는 새 모델 family 확보라는 관점에서는 의미가 있지만, raw frozen feature + 선형 meta-model 조합은 현재 ConvNeXt/EfficientNet OOF pool보다 약했습니다. 상위 후보가 보호 샘플을 여러 개 바꾸고 EXP-56 대비 변경 수도 커서 제출하지 않았습니다.


In [ ]:
# EXP-69 산출물 확인용 코드
from pathlib import Path
import json
import pandas as pd

result_path = Path('oof/preds/exp69_dinov2_vits14_result.json')
summary_path = Path('oof/preds/exp69_dinov2_vits14_summary.csv')

with result_path.open() as f:
    result = json.load(f)

summary = pd.read_csv(summary_path)
print('model:', result['model'])
print('feature_dim:', result['feature_dim'])
print('train_features:', result['train_features'])
print('test_features:', result['test_features'])
print('candidate_count:', result['candidate_count'])
print('saved_unique_submissions:', result['saved_unique_submissions'])
print('max_meta_cv_accuracy:', round(summary['meta_cv_accuracy'].max(), 5))

cols = [
    'model', 'param', 'threshold', 'meta_cv_accuracy', 'meta_cv_precision',
    'meta_cv_recall', 'meta_cv_f1', 'test_label1', 'changed_vs_current_best',
    'passes_protected', 'passes_label_band', 'candidate'
]
summary.sort_values('meta_cv_accuracy', ascending=False).head(8)[cols]


### 13-44. EXP-70 EfficientNetV2-S 재설계 5-fold OOF

EXP-42 EfficientNetV2-S는 `features[7] + classifier`만 학습해 validation accuracy가 낮고 FN이 크게 증가했습니다. EXP-70에서는 학습 범위를 `features[6:] + classifier`로 넓혀 EfficientNetV2-S가 OOF 정보원으로 쓸 수 있는지 재평가했습니다.

핵심 결과는 다음과 같습니다.

- OOF accuracy: 0.97642
- OOF precision: 0.99110
- OOF recall: 0.97703
- OOF F1: 0.98402
- OOF confusion matrix: `[[1307, 34], [89, 3786]]`
- test label 0/1: 208 / 416
- EXP-56 대비 변경 수: 34

해석: EXP-42 대비 성능은 크게 회복했으므로 마지막 block만 학습한 설정이 문제였을 가능성이 큽니다. 다만 단독 mean은 보호 샘플을 여러 개 깨므로 제출하지 않고, 이후 stacking 보조 feature로만 검토합니다.


In [ ]:
# EXP-70 산출물 확인용 코드
from pathlib import Path
import json
import pandas as pd

result_path = Path('oof/preds/exp70_efficientnet_v2_s_wide_combined_result.json')
changed_path = Path('oof/preds/exp70_changed_vs_exp56.csv')
submission_path = Path('oof/submissions/exp70_efficientnet_v2_s_wide_mean_submit.csv')

with result_path.open() as f:
    result = json.load(f)

submission = pd.read_csv(submission_path)
changed = pd.read_csv(changed_path)

print('OOF metrics:', result['oof_metrics'])
print('test_mean_pred_counts:', result['test_mean_pred_counts'])
print('changed_vs_exp56:', len(changed))
print('submission shape:', submission.shape)
print('submission labels:', sorted(submission['label'].unique()))

pd.DataFrame(result['sensitive']).T


### 13-45. EXP-71 EfficientNetV2-S 포함 OOF stacking

EXP-70 EfficientNetV2-S OOF를 기존 ConvNeXt/ConvNeXt seed123/EfficientNet-B0 OOF feature에 추가해 Ridge/Logistic stacking 후보를 탐색했습니다.

핵심 결과는 다음과 같습니다.

- 탐색 row 수: 6,318
- 최고 meta-CV accuracy: 0.98888
- 보호 샘플 8개 유지 row 수: 5
- 보호 샘플 유지 비중복 candidate: 0개
- 제출 파일: 없음

해석: meta-CV 최고값은 높았지만 상위 후보가 `test_0490`, `test_0504`, `test_0539`, `test_0244` 등 보호 방향을 자주 깼습니다. 보호 조건을 통과한 후보는 모두 EXP-56과 동일 예측이어서 새 제출 후보로 저장하지 않았습니다.


In [ ]:
# EXP-71 산출물 확인용 코드
import pandas as pd

summary = pd.read_csv('oof/preds/exp71_effv2_stacking_summary.csv')
print('rows:', len(summary))
print('candidate:', int(summary['candidate'].sum()))
print('passes_protected:', int(summary['passes_protected'].sum()))
print('max_meta_cv_accuracy:', round(summary['meta_cv_accuracy'].max(), 5))

cols = [
    'spec', 'model', 'param', 'threshold', 'meta_cv_accuracy',
    'test_label1', 'changed_vs_current_best', 'passes_protected',
    'passes_label_band', 'passes_meta_cv_floor', 'candidate'
]
summary.head(10)[cols]


### 13-46. EXP-72 ConvNeXt-Tiny seed777 5-fold OOF

ConvNeXt 계열은 현재까지 가장 강한 모델 family이므로, seed diversity 확보를 위해 seed777로 5-fold OOF를 추가 학습했습니다.

핵심 결과는 다음과 같습니다.

- OOF accuracy: 0.98006
- OOF precision: 0.98809
- OOF recall: 0.98503
- OOF F1: 0.98656
- OOF confusion matrix: `[[1295, 46], [58, 3817]]`
- test label 0/1: 210 / 414
- EXP-56 대비 변경 수: 20

해석: 단독 OOF 성능은 강하지만, test mean은 보호 샘플을 여러 개 1로 바꾸므로 제출하지 않습니다. 다음 단계에서 기존 ConvNeXt seed들과 함께 stacking feature로 사용합니다.


In [ ]:
# EXP-72 산출물 확인용 코드
from pathlib import Path
import json
import pandas as pd

result_path = Path('oof/preds/exp72_convnext_tiny_seed777_combined_result.json')
changed_path = Path('oof/preds/exp72_changed_vs_exp56.csv')
submission_path = Path('oof/submissions/exp72_convnext_tiny_seed777_mean_submit.csv')

with result_path.open() as f:
    result = json.load(f)

submission = pd.read_csv(submission_path)
changed = pd.read_csv(changed_path)

print('OOF metrics:', result['oof_metrics'])
print('test_mean_pred_counts:', result['test_mean_pred_counts'])
print('changed_vs_exp56:', len(changed))
print('submission shape:', submission.shape)
print('submission labels:', sorted(submission['label'].unique()))

pd.DataFrame(result['watched_samples']).T


### 13-47. EXP-73 ConvNeXt seed777 포함 OOF stacking

EXP-72 ConvNeXt seed777 OOF를 기존 ConvNeXt seed42, ConvNeXt seed123, EfficientNet-B0 OOF pool에 추가해 Ridge/Logistic stacking 후보를 탐색했습니다.

핵심 결과는 다음과 같습니다.

- 탐색 row 수: 20,592
- 최고 meta-CV accuracy: 0.98754
- 보호 샘플 8개 유지 row 수: 21
- 보호 샘플 유지 비중복 candidate: 0개
- 제출 파일: 없음

해석: seed777 자체는 강한 OOF 정보원이지만, 보호 샘플을 모두 유지하는 후보는 EXP-56과 동일 예측이었습니다. 상위 후보는 `test_0504`, `test_0490`, `test_0539`를 자주 1로 바꿔 public guardrail과 충돌했습니다.


In [ ]:
# EXP-73 산출물 확인용 코드
import pandas as pd

summary = pd.read_csv('oof/preds/exp73_convnext_seed_stacking_summary.csv')
print('rows:', len(summary))
print('candidate:', int(summary['candidate'].sum()))
print('passes_protected:', int(summary['passes_protected'].sum()))
print('protected non-duplicate:', int((summary['passes_protected'] & summary['not_duplicate_current']).sum()))
print('max_meta_cv_accuracy:', round(summary['meta_cv_accuracy'].max(), 5))

cols = [
    'spec', 'model', 'param', 'threshold', 'meta_cv_accuracy',
    'test_label1', 'changed_vs_current_best', 'passes_protected',
    'passes_label_band', 'passes_meta_cv_floor', 'candidate'
]
summary.head(10)[cols]


### 13-48. EXP-77 ConvNeXt-Tiny Lion fold0 smoke

ConvNeXt-Tiny fold0에서 optimizer만 AdamW에서 Lion으로 바꿔 smoke test를 진행했습니다. 목적은 전체 5-fold를 돌리기 전에 Lion이 학습 안정성과 오류 방향을 바꿀 수 있는지 확인하는 것입니다.

핵심 결과는 다음과 같습니다.

- Lion fold0 accuracy: 0.98563
- Lion fold0 recall: 0.98581
- Lion confusion matrix: `[[265, 4], [11, 764]]`
- AdamW fold0 reference accuracy: 0.98467
- AdamW fold0 reference recall: 0.98323
- test label 0/1: 234 / 390
- EXP-56 대비 변경 수: 26

해석: fold0 기준 Lion은 AdamW보다 accuracy와 recall이 소폭 개선됐습니다. 다만 `test_0539`, `test_0490`, `test_0141` 보호 방향을 깨므로 단독 제출하지 않고, 5-fold OOF 확장 후보로만 기록합니다.


In [ ]:
# EXP-77 산출물 확인용 코드
from pathlib import Path
import json
import pandas as pd

result_path = Path('oof/preds/exp77_convnext_tiny_lion_fold0_result.json')
changed_path = Path('oof/preds/exp77_changed_vs_exp56.csv')
submission_path = Path('oof/submissions/exp77_convnext_tiny_lion_fold0_submit.csv')

with result_path.open() as f:
    result = json.load(f)

submission = pd.read_csv(submission_path)
changed = pd.read_csv(changed_path)

print('Lion final metrics:', result['final_metrics'])
print('AdamW fold0 reference:', result['adamw_fold0_reference'])
print('test_pred_counts:', result['test_pred_counts'])
print('changed_vs_exp56:', len(changed))
print('submission shape:', submission.shape)

pd.DataFrame(result['watched_samples']).T


## EXP-78. ConvNeXt-Tiny Lion 5-fold OOF 복구 기록

로컬 Docker/IDE 메모리 문제로 EXP-78이 fold2 진행 중 중단되었지만, fold0/fold1 산출물은 정상 저장되어 있었다. 이후 `scripts/run_exp78_convnext_lion_oof.py`를 fold2~4만 재실행하도록 수정하고, fold0~4 전체를 병합해 5-fold OOF 결과를 완성했다.

- 모델: ConvNeXt-Tiny ImageNet pretrained
- 학습 범위: 마지막 stage `features[-1] + classifier`
- optimizer: Lion
- LR: `2e-5`
- weight decay: `1e-4`
- OOF accuracy: `0.98140`
- OOF confusion matrix: `[[1294, 47], [50, 3825]]`
- test label 0/1: `204 / 420`
- EXP-56 대비 변경 수: `24`

보호 샘플 기준으로 `test_0025=0`, `test_0201=1`, `test_0222=1`, `test_0141=1`, `test_0568=0`은 유지했지만, `test_0539`, `test_0244`, `test_0490`, `test_0504`, `test_0188`을 1로 바꾸므로 단독 제출은 제외한다. 다만 FN을 줄이는 경향이 있어 이후 Ridge/Logistic stacking의 보조 feature로 활용할 수 있다.

생성 파일:

```text
scripts/run_exp78_convnext_lion_oof.py
oof/preds/exp78_convnext_tiny_lion_oof_probs.csv
oof/preds/exp78_convnext_tiny_lion_test_probs_by_fold.csv
oof/preds/exp78_convnext_tiny_lion_test_probs_mean.csv
oof/preds/exp78_convnext_tiny_lion_fold_summary.csv
oof/preds/exp78_convnext_tiny_lion_combined_result.json
oof/preds/exp78_changed_vs_exp56.csv
oof/submissions/exp78_convnext_tiny_lion_mean_submit.csv
```


## EXP-79. Lion OOF 포함 stacking

EXP-78 ConvNeXt-Tiny Lion OOF를 기존 ConvNeXt/EfficientNet OOF pool에 추가해 Ridge/Logistic stacking을 탐색했다. 목적은 Lion 단독 모델을 제출하는 것이 아니라, optimizer가 다른 ConvNeXt의 오류 방향을 meta-model에서 보완 정보로 활용할 수 있는지 확인하는 것이었다.

결과:

- 탐색 row 수: `40,248`
- 전체 최고 meta-CV accuracy: `0.98811`
- 보호 샘플 유지 row 수: `32`
- 최종 candidate 수: `0`
- 제출 파일: 없음

가장 가까운 보호 후보는 `cnx_lion_eff_prob_only`, Ridge alpha `500`, threshold `0.20` 조합이었다. 이 후보는 EXP-56 대비 `test_0049: 0 -> 1` 한 개만 바꾸고 보호 샘플은 유지했지만, meta-CV accuracy가 `0.98447`로 현재 기준 `0.98658`보다 약 `0.00211` 낮았다.

해석: Lion feature는 일부 조합에서 meta-CV를 올렸지만, 그 후보들은 보호 샘플을 깨는 경향이 있었다. 보호 샘플을 유지하는 후보는 내부 검증 성능 하락폭이 커서 제출하지 않는다.

생성 파일:

```text
scripts/run_exp79_lion_stacking.py
oof/preds/exp79_lion_stacking_summary.csv
```


### EXP-79 추가 제출 후보 생성

사용자 요청으로 EXP-79에서 가장 가까운 보호 후보를 제출 CSV로 별도 생성했다.

- 파일: `oof/submissions/exp79_ridge_cnx_lion_eff_prob_only_alpha500_t0p2_submit.csv`
- 모델 조합: ConvNeXt 기본 + ConvNeXt Lion + EfficientNet-B0 probability only
- meta-model: RidgeClassifier alpha `500`
- threshold: `0.20`
- test label 0/1: `227 / 397`
- EXP-56 대비 변경 수: `1`
- 변경 샘플: `test_0049: 0 -> 1`
- 보호 샘플: 모두 유지

내부 기준으로는 meta-CV accuracy가 `0.98447`로 현재 기준 `0.98658`보다 낮아 우선순위가 높지는 않다. 다만 변경 수가 1개이고 보호 샘플을 모두 유지하므로, 탐색 제출 후보로는 가장 보수적인 선택이다.


### EXP-79 제출 결과

`oof/submissions/exp79_ridge_cnx_lion_eff_prob_only_alpha500_t0p2_submit.csv`를 제출했다.

- Leaderboard score: `0.9567307692`
- EXP-56 최고점: `0.9583333333`
- EXP-56 대비 변경: `test_0049: 0 -> 1`

결과적으로 `test_0049=1` 방향은 public 기준에서 이득이 아니었다. 이후 low-change 후보를 만들 때는 `test_0049=0`을 우선 유지하는 편이 낫다.


## EXP-80. ResNet18 제한 포함 compact stacking

EXP-56 OOF pool에 ResNet18 OOF를 제한적으로 다시 추가해 Ridge/Logistic stacking을 탐색했다. EXP-79 제출 결과를 반영해 `test_0049=0`도 보호 조건에 추가했다.

결과:

- 탐색 row 수: `21,060`
- 전체 최고 meta-CV accuracy: `0.98907`
- 보호 샘플 유지 row 수: `21`
- 최종 candidate 수: `0`
- 제출 파일: 없음

전체 최고 후보는 `all_prob_stats` Ridge alpha `10`, threshold `0.05` 조합이었다. 하지만 이 후보는 EXP-56 대비 16개를 바꾸고, `test_0049`, `test_0504`, `test_0539`, `test_0244`, `test_0490`, `test_0188`을 모두 1로 바꿔 보호 조건과 충돌했다.

해석: ResNet18을 추가하면 내부 meta-CV는 올라갈 수 있지만, public에서 확인된 guardrail과 충돌하는 방향이 강했다. 현재 기준에서는 새 제출 후보 없이 보류한다.

생성 파일:

```text
scripts/run_exp80_resnet_compact_stacking.py
oof/preds/exp80_resnet_compact_stacking_summary.csv
```


### EXP-65 단독 mean 제출 결과

코덱스 사용 중단 기간에 `oof/submissions/exp65_convnextv2_tiny_stage3_mean_submit.csv`를 별도로 제출했다.

- Leaderboard score: `0.9262820513`
- OOF accuracy: `0.97680`
- EXP-56 최고점: `0.9583333333`

ConvNeXt V2 Tiny 단독 mean은 `test_0025`, `test_0539`, `test_0244`, `test_0490`, `test_0504` 등 보호 방향을 다수 깨고 label 1을 416개로 많이 예측했다. 실제 public 점수도 크게 하락했으므로, 이 모델은 단독 제출용이 아니라 stacking diversity feature로만 제한적으로 활용한다.


## EXP-81. EfficientNet-B0 seed123 5-fold OOF

EXP-50과 동일한 EfficientNet-B0 구조와 fold split을 유지하고 학습 seed만 123으로 변경했다. 목적은 같은 family 안에서 오류 방향이 다른 OOF 정보원을 확보하는 것이다.

결과:

- OOF accuracy: `0.98332`
- OOF precision: `0.99272`
- OOF recall: `0.98477`
- OOF F1: `0.98873`
- confusion matrix: `[[1313, 28], [59, 3816]]`
- test label 0/1: `251 / 373`
- EXP-50 대비 OOF/test label disagreement: `45 / 14`
- EXP-50 대비 OOF/test probability correlation: `0.99227 / 0.99217`
- EXP-56 대비 test 변경 수: `29`

단독 mean은 `test_0201`, `test_0222`, `test_0141`, `test_0568`의 보호 방향을 깨므로 제출하지 않는다. 다만 EXP-50과 test 예측 14개가 달라 EXP-82에서 낮은 비중의 stacking feature로 검토할 가치가 있다.

생성 파일:

```text
scripts/run_exp81_efficientnet_b0_seed123_oof.py
oof/preds/exp81_efficientnet_b0_seed123_oof_probs.csv
oof/preds/exp81_efficientnet_b0_seed123_test_probs_by_fold.csv
oof/preds/exp81_efficientnet_b0_seed123_test_probs_mean.csv
oof/preds/exp81_efficientnet_b0_seed123_fold_summary.csv
oof/preds/exp81_efficientnet_b0_seed123_combined_result.json
oof/preds/exp81_changed_vs_exp56.csv
oof/submissions/exp81_efficientnet_b0_seed123_mean_submit.csv
```


## EXP-82~91. 빠른 OOF 분석 실험 모음

기존 OOF/TTA 확률을 재사용해 재학습 없이 10개 실험을 수행했다.

| EXP | 실험 | 결과 |
|---|---|---|
| 82 | EfficientNet seed stacking | 최고 meta-CV 0.98735, 보호 후보 0개 |
| 83 | EfficientNet seed soft voting | 최고 OOF 0.98581, 보호 위반 |
| 84 | 오류 상관관계 | ConvNeXt-ResNet disagreement 126개 |
| 85 | Ridge feature importance | eff > eff123 > cnx123 > cnx |
| 86 | Feature pruning | cnx123 제거 시 OOF 0.98658 |
| 87 | Threshold/calibration | 최고 OOF 0.98696, 보호 후보 없음 |
| 88 | Low-change | 162개 중 보호 통과 0개 |
| 89 | FP/FN 전문화 | ResNet FN 40으로 가장 적음 |
| 90 | Rank ensemble | 최고 OOF 0.98658, 변경 수 과다 |
| 91 | 기존 TTA 재분석 | 단일 validation 기반이라 분석용 |

새 제출 후보는 생성되지 않았다. EfficientNet 계열은 stacking 기여도가 가장 높고, ResNet은 FN 감소, ConvNeXt는 precision 중심의 보수적 역할을 보였다. 미세 threshold 탐색은 상당히 소진됐으므로 다음에는 새로운 모델 family 또는 표현 공간을 확보하는 편이 타당하다.

생성 파일:

```text
scripts/run_exp82_91_fast_suite.py
oof/preds/exp82_83_fast_search_summary.csv
oof/preds/exp84_oof_diversity_summary.csv
oof/preds/exp85_ridge_feature_importance.csv
oof/preds/exp86_feature_pruning_summary.csv
oof/preds/exp87_threshold_calibration_summary.csv
oof/preds/exp88_low_change_summary.csv
oof/preds/exp89_fp_fn_specialization.csv
oof/preds/exp90_rank_ensemble_summary.csv
oof/preds/exp91_existing_tta_direction_summary.csv
```


## EXP-92. DINOv2 PCA feature 개선

EXP-69에서 저장한 frozen DINOv2 384차원 feature를 재사용하고, fold 내부에서 PCA를 fit해 노이즈를 줄였다.

- PCA: 32/64/128/256/no PCA
- model: RidgeClassifier / LogisticRegression
- 최고 조합: PCA 256 + LogisticRegression C=0.05, threshold -0.20
- meta-CV accuracy: `0.97604`
- EXP-69 대비 개선: 약 `+0.00230`
- confusion matrix: `[[1286, 55], [70, 3805]]`
- EXP-56 대비 test 변경: `71개`
- 보호 조건 통과 후보: `0개`

PCA를 통한 표현 압축은 DINOv2 성능을 소폭 개선했지만, 현재 CNN OOF pool보다 낮고 test 변경 수가 많아 제출 후보로는 사용하지 않는다.

```text
scripts/run_exp92_dinov2_pca_search.py
oof/preds/exp92_dinov2_pca_summary.csv
oof/preds/exp92_dinov2_pca_result.json
```


## EXP-93. ConvNeXt vs ResNet 판단 방향 차이 분석

두 모델의 OOF disagreement 126개를 대상으로 밝기, 대비, entropy, gradient, 중앙/주변 명암을 비교했다.

- ConvNeXt=0, ResNet=1: `90개`
- ConvNeXt=1, ResNet=0: `36개`
- 영상 통계 기반 방향 분류 accuracy: `0.532`
- balanced accuracy: `0.514`

단순 영상 통계만으로 어느 모델이 1을 예측할지 구분하는 성능이 거의 우연 수준이었다. 따라서 밝기나 대비보다 모델이 학습한 공간적 표현 차이가 더 유력하다.

`ConvNeXt=0, ResNet=1` 그룹은 실제 양성 비율이 `71.1%`였고 ResNet 정확도도 `71.1%`였다. 이는 ResNet의 FN이 ConvNeXt보다 적었던 결과와 일치하지만, 이 결과만으로 ResNet이 실제 병변을 보았다고 단정할 수는 없다.

### 대표 이미지: ConvNeXt=0, ResNet=1

![CNX0 RES1](reports/exp93_disagreement/cnx0_res1_representatives.png)

### 대표 이미지: ConvNeXt=1, ResNet=0

![CNX1 RES0](reports/exp93_disagreement/cnx1_res0_representatives.png)

다음 설명 실험은 동일 샘플에 대한 fold별 Grad-CAM 비교다.


## EXP-94. OOF disagreement Grad-CAM 비교

각 샘플이 validation이었던 fold checkpoint를 사용해 ConvNeXt와 ResNet의 class-1 Grad-CAM을 비교했다.

- 대표 샘플: `8장`
- CAM pixel correlation 평균: `0.251`
- CAM 상위 20% 영역 IoU 평균: `0.216`
- CNX=0/RES=1에서 중앙 CAM 비중: ConvNeXt `0.272`, ResNet `0.379`
- CNX=1/RES=0에서 ResNet 상단 CAM 비중: `0.754`

두 모델은 같은 이미지에서도 class 1 근거로 반응하는 위치가 상당히 달랐다. 이는 방향성 차이가 단순 밝기보다 공간적 표현 차이에서 발생한다는 EXP-93 결과를 보완한다. 다만 낮은 확률 class의 CAM은 불안정할 수 있고, 반응 위치를 실제 병변이라고 단정할 수 없다.

![OOF Grad-CAM comparison](reports/exp94_gradcam/oof_gradcam_disagreement_panel.png)


## EXP-95. Suspicious test case review pack

leaderboard score 변화로 test 정답을 역추론하는 방식은 피하고, 모델 간 불일치와 불확실성이 큰 test 이미지를 리뷰 대상으로 묶었다.

- 기준 제출: `oof/submissions/exp56_ridge_cnx_eff_compact_p3000_t0p025_submit.csv`
- 사용 모델 수: `10`
- action count: keep `553`, review `60`, avoid_change_protected `9`, review_high_priority `2`
- high-priority review 후보: `test_0188.png`, `test_0377.png`
- 보호 샘플은 `avoid_change_protected`로 분리해 임의 라벨 변경 대상에서 제외

산출물:

```text
scripts/run_exp95_suspicious_test_pack.py
reports/exp95_suspicious_cases/exp95_suspicious_cases.csv
reports/exp95_suspicious_cases/exp95_suspicious_top50.csv
reports/exp95_suspicious_cases/exp95_suspicious_panel_top50.png
reports/exp95_suspicious_cases/report.md
```

해석: `suspicious_score`는 라벨 변경 지시가 아니라 XAI/오류 분석 우선순위이다. 의료 이미지에서는 이미지 패널이나 XAI 반응을 실제 병변 근거로 단정하지 않고, 모델 반응 위치와 예측 불확실성으로만 해석한다.


## EXP-96. Suspicious rule submission candidates

EXP-95의 `review_high_priority` 샘플만 사용해 보호 샘플을 유지하는 제출 후보를 생성했다. 이는 수동 라벨 추측이 아니라 모델 간 불일치와 EXP-56 decision margin을 사용한 규칙 기반 후보이다.

제출 우선순위:

1. `oof/submissions/exp96_flip0188_only_submit.csv`
   - 변경: `test_0188: 0 -> 1`
   - 근거: EXP-56 margin `-0.0109`, 10개 모델 중 7개가 1
2. `oof/submissions/exp96_flip0377_only_submit.csv`
   - 변경: `test_0377: 0 -> 1`
   - 근거: 10개 모델 중 6개가 1
3. `oof/submissions/exp96_flip0188_0377_submit.csv`
   - 변경: 2개라 공격적 후보

검증:

- row 수 624
- 컬럼 `file_name,label`
- sample submission과 file order 일치
- label은 0/1만 존재
- 보호 샘플 9개 모두 유지

제출 결과:

- 제출 파일: `oof/submissions/exp96_flip0188_only_submit.csv`
- Leaderboard score: `0.9567307692`
- EXP-56 score: `0.9583333333`
- 변경: `test_0188: 0 -> 1`
- 결론: EXP-56보다 하락했으므로 이후 `test_0188=0` 유지 우선


## EXP-97. Guardrail tree filter

EXP-95 suspicious ranking과 public 결과로 확인된 보호 방향을 이용해, 기존 제출 CSV 181개를 일괄 평가하는 guardrail tree를 만들었다.

핵심 규칙:

- hard guardrail: `test_0025=0`, `test_0049=0`, `test_0141=1`, `test_0188=0`, `test_0201=1`, `test_0222=1`, `test_0244=0`, `test_0490=0`, `test_0504=0`, `test_0539=0`
- conditional guardrail: `test_0188=1`과 `test_0568=1` 동시 변경 금지
- label1 hard range: 392~402
- label1 soft range: 395~397
- EXP-95 `avoid_change_protected` 변경 금지
- EXP-95 `review_high_priority` 변경은 새 근거 없으면 fail

결과:

- 평가 파일 수: `181`
- pass: `2`
- fail: `179`
- pass 파일은 `exp55_ridge_cnx_eff_compact_p3000_t0p025_submit.csv`, `exp56_ridge_cnx_eff_compact_p3000_t0p025_submit.csv`
- 두 파일 모두 EXP-56과 동일 예측이므로 새 제출 후보는 없음

산출물:

```text
scripts/run_exp97_guardrail_tree_filter.py
oof/preds/exp97_guardrail_tree_report.csv
oof/preds/exp97_guardrail_tree_passed.csv
oof/preds/exp97_guardrail_tree.md
```

해석: guardrail tree는 제출 추천기가 아니라 제출 전 위험 필터이다. 현재까지 하락한 후보 대부분은 보호 샘플 변경을 포함했으므로, 새 모델/stacking 후보가 생겼을 때 이 필터를 먼저 통과시키는 방식이 타당하다.


## EXP-98. New model family precheck

ConvNeXt/EfficientNet 중심 OOF pool이 포화되어, 다른 오류 방향을 줄 수 있는 새 모델 family 후보를 사전 검토했다. 바로 5-fold 학습하지 않고 timm 지원 여부, 파라미터 수, smoke 우선순위를 확인했다.

환경:

- torch `2.12.0`
- torchvision `0.27.0`
- timm `1.0.27`
- device `mps`

후보 순위:

| 순위 | 모델 | total params | trainable guess | 판단 |
|---:|---|---:|---:|---|
| 1 | `maxvit_tiny_rw_224.sw_in1k` | 28,545,338 | 16,336,130 | fold0 smoke 2 epochs 1순위 |
| 2 | `coatnet_nano_rw_224.sw_in1k` | 14,629,270 | 8,874,346 | MaxViT 실패 시 가벼운 대안 |
| 3 | `swinv2_tiny_window8_256.ms_in1k` | 27,579,692 | 15,384,626 | window attention diversity 후보 |
| 4 | `deit3_small_patch16_224.fb_in1k` | 21,675,266 | 1,776,002 | 순수 ViT supervised fine-tuning 후보 |
| 5 | `coatnet_0_rw_224.sw_in1k` | 26,668,100 | 12,854,342 | smoke 후 장기 후보 |

다음 실행 게이트:

- 바로 5-fold 학습하지 않는다.
- `maxvit_tiny_rw_224.sw_in1k` fold0 smoke 2 epoch부터 확인한다.
- validation metric, confusion matrix, EXP-56 대비 test 변경 수, EXP-97 guardrail 위반 여부를 본다.
- smoke가 보호 샘플을 대량으로 깨면 5-fold 확장을 중단한다.

산출물:

```text
scripts/run_exp98_model_family_precheck.py
reports/exp98_model_family_precheck/exp98_model_family_precheck.csv
reports/exp98_model_family_precheck/exp98_model_family_details.json
reports/exp98_model_family_precheck/report.md
```


## EXP-99. MaxViT Tiny fold0 smoke

EXP-98의 1순위 후보인 `maxvit_tiny_rw_224.sw_in1k`를 fold0 2 epoch smoke로 검증했다. 바로 5-fold로 확장하지 않고 validation 성능과 EXP-97 guardrail 충돌 여부를 확인하는 목적이다.

설정:

- 모델: `maxvit_tiny_rw_224.sw_in1k`
- 학습 범위: `stages[-1] + head`
- trainable params: `16,336,130`
- epochs: `2`
- batch size: `16`
- optimizer: AdamW
- LR: stage `2e-5`, head `8e-5`

결과:

- fold0 valid accuracy: `0.96552`
- precision / recall / F1: `0.99332 / 0.96000 / 0.97638`
- confusion matrix: `[[264, 5], [31, 744]]`
- test label 0/1: `228/396`
- EXP-56 대비 변경 수: `38`
- guardrail fail count: `7`

Guardrail 실패:

```text
test_0025: 0 -> 1
test_0049: 0 -> 1
test_0141: 1 -> 0
test_0188: 0 -> 1
test_0244: 0 -> 1
test_0490: 0 -> 1
test_0504: 0 -> 1
```

판단: 단독 제출 및 즉시 5-fold 확장은 보류한다. MaxViT가 기존 모델과 다른 방향의 신호를 줄 수는 있지만, 현재 smoke 기준으로는 보호 샘플 충돌이 크므로 낮은 가중치 stacking feature 또는 오류 분석용으로만 검토한다.


## EXP-100. CoAtNet Nano fold0 smoke

EXP-98의 2순위 후보인 `coatnet_nano_rw_224.sw_in1k`를 fold0 2 epoch smoke로 검증했다. MaxViT보다 가벼운 hybrid 모델이 guardrail을 덜 깨는지 확인하는 목적이다.

설정:

- 모델: `coatnet_nano_rw_224.sw_in1k`
- 학습 범위: `stages[-1] + head`
- trainable params: `8,874,346`
- epochs: `2`
- batch size: `16`
- optimizer: AdamW
- LR: stage `2e-5`, head `8e-5`

결과:

- fold0 valid accuracy: `0.96456`
- precision / recall / F1: `0.99200 / 0.96000 / 0.97574`
- confusion matrix: `[[263, 6], [31, 744]]`
- test label 0/1: `316/308`
- EXP-56 대비 변경 수: `90`
- guardrail fail count: `4`

Guardrail 실패:

```text
test_0141: 1 -> 0
test_0188: 0 -> 1
test_0201: 1 -> 0
test_0222: 1 -> 0
```

판단: MaxViT보다 guardrail fail 수는 적지만, test label1이 308개로 지나치게 보수적이다. EXP-56 대비 변경도 90개로 많아 단독 제출 및 즉시 5-fold 확장은 보류한다.

리더보드 결과:

- 제출 파일: `oof/submissions/exp100_coatnet_nano_fold0_smoke_submit.csv`
- Public score: `0.8493589744`
- EXP-56 score: `0.9583333333`
- 차이: `-0.1089743589`

해석: fold0 validation accuracy는 `0.96456`으로 완전히 낮지는 않았지만, public score는 크게 하락했다. 사전 경고였던 test label1 `308`, EXP-56 대비 변경 `90`개, 보호 샘플 실패 `4`개가 실제 generalization 위험으로 확인되었다. 따라서 CoAtNet Nano 단독 제출과 5-fold 확장은 제외한다.


## EXP-101~104. Gemini 산출물 로컬 검증

이 구간은 Gemini 세션에서 생성된 산출물을 현재 로컬에서 검증한 기록이다. 직접 실행 완료 실험으로 단정하지 않고 `Gemini generated / locally verified` 상태로 분리한다.

검증 항목:

- EXP-101/102 제출 CSV: 624 rows, columns `file_name,label`, sample submission order 일치
- label 값: 0/1만 존재
- JSON 결과 파일: 파싱 통과
- EXP-101~104 스크립트: `py_compile` 통과

EXP-101 SwinV2-Tiny fold0 smoke:

- model: `swinv2_tiny_window8_256.ms_in1k`
- fold0 valid accuracy: `0.97414`
- precision / recall / F1: `0.97826 / 0.98710 / 0.98266`
- confusion matrix: `[[252, 17], [10, 765]]`
- test label 0/1: `168/456`
- EXP-56 대비 변경 수: `68`
- guardrail fail count: `8`
- 판단: 단독 제출 금지. full OOF 완료 후 낮은 영향 stacking feature로만 검토한다.

EXP-102 DeiT-Small fold0 smoke:

- model: `deit3_small_patch16_224.fb_in1k`
- fold0 valid accuracy: `0.91858`
- precision / recall / F1: `0.98592 / 0.90323 / 0.94276`
- confusion matrix: `[[259, 10], [75, 700]]`
- test label 0/1: `427/197`
- EXP-56 대비 변경 수: `213`
- guardrail fail count: `3`
- 판단: 단독 제출 및 5-fold 확장 우선순위가 낮다.

EXP-103 SwinV2-Tiny 5-fold OOF:

- Gemini partial 산출물에서 fold0/fold1 재사용, fold2~4 로컬 실행 완료
- fold0 valid accuracy: `0.97414`
- fold1 valid accuracy: `0.97124`
- fold2 valid accuracy: `0.95973`
- fold3 valid accuracy: `0.97315`
- fold4 valid accuracy: `0.97124`
- OOF accuracy / precision / recall / F1: `0.96990 / 0.98411 / 0.97523 / 0.97965`
- OOF confusion matrix: `[[1280, 61], [96, 3779]]`
- test mean label 0/1: `197/427`
- EXP-56 대비 변경 수: `53`
- guardrail fail count: `7`
- 판단: 단독 mean 제출은 금지하고, EXP-104 stacking feature로만 사용한다.

EXP-104 SwinV2 regularized stacking:

- 실행 완료
- 탐색 row 수: `2520`
- SwinV2 포함 row 수: `1440`
- strict candidate 수: `0`
- relaxed candidate 수: `0`
- 보호 샘플 전체 통과 row 수: `0`
- label band 통과 row 수: `420`
- low-change 통과 row 수: `577`
- meta-CV floor 통과 row 수: `1191`
- 최고 SwinV2 포함 meta-CV: `0.98677`
- 판단: 모든 후보가 보호 샘플을 최소 1개 이상 위반했으므로 제출 후보 없음. 다음은 SwinV2 low-weight/pruned stacking 또는 SwinV2 제외 low-change 재탐색이 적합하다.


## EXP-106. No-Swin low-change search

SwinV2를 제외하고 ConvNeXt/EfficientNet/ResNet 계열 OOF probability만 사용해 EXP-56 대비 low-change 후보를 다시 탐색했다.

- 탐색 row 수: 21,780
- candidate 수: 0
- 보호 샘플 전체 통과 row 수: 34
- 최고 meta-CV: 0.98926
- 산출물:
  - `scripts/run_exp106_no_swin_low_change_search.py`
  - `oof/preds/exp106_no_swin_low_change_summary.csv`

해석: SwinV2를 제외해도 강한 후보들은 주로 `test_0504=1`로 되돌아가며 EXP-56의 public 개선 방향을 깨는 경향이 있었다. 보호 샘플을 모두 유지한 row는 EXP-56과 동일 예측이라 새 제출 파일을 만들지 않았다. 따라서 현재 OOF pool의 단순 Ridge/Logistic low-change 탐색은 거의 소진된 것으로 보고, 다음은 경계 샘플 분석 또는 새 정보원 확보가 더 타당하다.



## EXP-107. Boundary bottleneck analysis

EXP-106에서 제출 후보가 0개였으므로, label band / low-change / meta-CV floor / non-duplicate 조건을 통과한 plausible row 5,784개를 대상으로 보호 샘플 병목을 분석했다.

- 최대 병목: `test_0504`
  - 기대 label: 0
  - 위반 row: 5,383 / 5,784
  - 위반 비율: 0.93067
- 다음 병목: `test_0490`, `test_0539`, `test_0049`, `test_0188`
- 비보호 반복 변경 샘플: `test_0568` 1개
- 산출물:
  - `scripts/run_exp107_boundary_bottleneck_analysis.py`
  - `reports/exp107_boundary_bottleneck/exp107_boundary_bottleneck_report.md`
  - `reports/exp107_boundary_bottleneck/exp107_protected_bottleneck.csv`

해석: EXP-56은 `test_0504`를 1에서 0으로 바꿔 public 최고점을 만들었지만, EXP-106의 강한 후보 대부분은 이 샘플을 다시 1로 되돌린다. 따라서 현재 OOF pool에서는 meta-CV를 높이는 방향과 public에서 확인된 보호 방향이 충돌한다. 다음은 `test_0504`, `test_0490`, `test_0539`, `test_0049`, `test_0568`의 모델별 확률과 XAI 패널을 확인하는 것이 타당하다.



## EXP-108. Flip0568 rule candidates

EXP-107에서 비보호 반복 변경 샘플이 `test_0568` 하나로 좁혀졌기 때문에, EXP-56 winner 기준으로 작은 변경 제출 후보를 생성했다.

생성 후보:

1. `oof/submissions/exp108_flip0568_only_submit.csv`
   - EXP-56 대비 `test_0568: 0 -> 1`
   - test label 0/1: 227/397
   - 보호 샘플/avoid-pair 조건 통과

2. `oof/submissions/exp108_flip0377_only_recheck_submit.csv`
   - EXP-56 대비 `test_0377: 0 -> 1`
   - 비교용 후보

3. `oof/submissions/exp108_flip0568_0377_submit.csv`
   - EXP-56 대비 `test_0568`, `test_0377` 2개 변경
   - 공격적 비교 후보

1순위 제출 후보는 `exp108_flip0568_only_submit.csv`다. 다만 `test_0568`은 EXP-95에서 model vote 1이 2개뿐이고 recommended action도 keep이었으므로 기대 이득은 크지 않다. 점수 변화 폭은 최대 1/624 수준으로 본다.



### EXP-108 leaderboard result

`oof/submissions/exp108_flip0568_only_submit.csv`를 제출했다.

- 변경: EXP-56 대비 `test_0568: 0 -> 1`
- Leaderboard score: `0.9567307692`
- EXP-56 score: `0.9583333333`
- 결과: EXP-56보다 하락

해석: `test_0568` 단독 0->1 변경도 public에서는 이득이 아니었다. EXP-58에서 `test_0188`과 함께 1로 바꿨을 때도 하락했으므로, 이후 후보 필터링에서는 `test_0568=0`을 유지 우선 방향으로 둔다.



## EXP-109. Extended guardrail filter

EXP-108에서 `test_0568: 0 -> 1` 단독 변경도 leaderboard 하락으로 확인되었으므로, `test_0568=0`까지 hard guardrail에 추가해 기존 제출 CSV 전체를 다시 평가했다.

- 평가 파일 수: 189
- 최신 guardrail 통과 파일 수: 2
- EXP-56 대비 비중복 통과 후보: 0
- 통과 파일:
  - `oof/submissions/exp55_ridge_cnx_eff_compact_p3000_t0p025_submit.csv`
  - `oof/submissions/exp56_ridge_cnx_eff_compact_p3000_t0p025_submit.csv`
- 두 파일 모두 EXP-56과 동일 예측

해석: 기존 제출 후보 전체를 최신 보호 규칙으로 다시 걸러도 새 제출 후보는 남지 않았다. 기존 CSV 재활용 기반의 후보 탐색은 소진된 것으로 보고, 다음은 새 OOF 정보원 확보 또는 경계 샘플 XAI/확률 패널로 방향을 바꾼다.



## EXP-110~112. ConvNeXt hard-case drop audit

이수인님 일지의 ConvNeXt 확률 순위 + threshold + hard case drop 전략을 우리 산출물 기준으로 재현했다.

### EXP-110

EXP-56에서 label 1인 샘플 중 ConvNeXt 계열 확률이 서로 갈리거나 margin이 낮은 샘플을 정렬해 1개 drop 후보를 생성했다.

- `oof/submissions/exp110_drop_test_0172_rank1_submit.csv`: `test_0172: 1 -> 0`
- `oof/submissions/exp110_drop_test_0455_rank2_submit.csv`: `test_0455: 1 -> 0`
- `oof/submissions/exp110_drop_test_0057_rank3_submit.csv`: `test_0057: 1 -> 0`

### EXP-111

ConvNeXt score 분포에서 EXP-56 양성 중 score가 낮은 샘플을 threshold 상승으로 drop하는 후보를 생성했다.

- 1개 drop 후보: `test_0289`, `test_0455`, `test_0013`
- 2개 drop 후보: `test_0289+test_0385`, `test_0289+test_0455`, `test_0289+test_0303`, `test_0013+test_0172`

### EXP-112

EXP-110/111 후보 7개에 대해 ConvNeXt fold0 class-1 Grad-CAM 패널을 생성했다.

- 패널: `reports/exp112_drop_candidate_gradcam/exp112_drop_candidate_gradcam_panel.png`
- 요약: `reports/exp112_drop_candidate_gradcam/exp112_drop_candidate_gradcam_summary.csv`

판단: XAI 단독으로 정답을 확정하지 않는다. 다만 후보 우선순위 조정에는 활용한다. 현재 1순위 제출 후보는 `oof/submissions/exp110_drop_test_0455_rank2_submit.csv`이며, 동일 예측 파일은 `oof/submissions/exp111_core_min_drop1_submit.csv`다.



## EXP-110 leaderboard result: ConvNeXt hard-case FP drop

- 제출 파일: `oof/submissions/exp110_drop_test_0455_rank2_submit.csv`
- Public score: `0.9599358974`
- 이전 최고 EXP-56: `0.9583333333`
- 변화: `test_0455: 1 -> 0`, EXP-56 대비 `+1/624` 개선
- 2026-06-30 기준 순위: 12위

해석: ConvNeXt 계열에서 확률이 낮고 모델 간 불일치가 있었던 hard-case FP 후보 중 `test_0455` 단독 drop이 실제 public에서 이득으로 확인되었다. 이후 후보 생성에서는 `test_0455=0`을 새 보호 방향으로 유지한다. 기존 EXP-110/111 후보는 EXP-56 기준 파일이므로, 그대로 제출하면 `test_0455`를 다시 1로 되돌릴 수 있어 새 기준선 위에서 재생성해야 한다.


## EXP-111 leaderboard result: current-best + test_0289 drop

- 제출 파일: `oof/submissions/exp111_core_min_drop2_submit.csv`
- Public score: `0.9583333333`
- 현재 최고 EXP-110: `0.9599358974`
- 현재 최고 대비 변경: `test_0289: 1 -> 0`

해석: `test_0455=0`은 유지했지만, `test_0289`를 추가로 0으로 바꾸면 public 점수가 하락했다. 따라서 이후 후보 생성에서는 `test_0289=1`을 우선 보호 방향으로 유지하고, `test_0057`, `test_0172`, `test_0013`, `test_0385`, `test_0303` 등 다른 hard-case 후보를 현재 최고 EXP-110 기준에서 하나씩만 추가 변경하는 방식으로 검토한다.


## EXP-113~114. Binary-tree replan and current-best single-drop candidates

### EXP-113 evidence table

이전 EXP-95/97의 교훈은 suspicious score만으로는 불확실한 샘플은 찾을 수 있지만 정답 방향까지 잘 맞추지는 못했다는 점이다. 그래서 EXP-110 성공(`test_0455=0`)과 EXP-111 실패(`test_0289=0`)를 반영해 current-best 기준 evidence table을 다시 만들었다.

- 기준 파일: `oof/submissions/exp110_drop_test_0455_rank2_submit.csv`
- 유지 방향: `test_0455=0`, `test_0289=1`
- 산출물: `reports/exp113_tree_replan/exp113_candidate_evidence_table.csv`
- ranking: `reports/exp113_tree_replan/exp113_candidate_ranked.csv`

### EXP-114 제출 후보

기존 EXP-110/111 CSV를 그대로 쓰지 않고, current-best 위에서 정확히 1개만 `1->0`으로 바꾸는 파일을 새로 만들었다. 모든 후보는 624행, `file_name,label` 컬럼, `test_0455=0`, `test_0289=1` 조건을 통과했다.

| priority | file | change | label 0/1 |
|---:|---|---|---:|
| 1 | `oof/submissions/exp114_drop_test_0057_only_submit.csv` | `test_0057: 1->0` | 230/394 |
| 2 | `oof/submissions/exp114_drop_test_0172_only_submit.csv` | `test_0172: 1->0` | 230/394 |
| 3 | `oof/submissions/exp114_drop_test_0385_only_submit.csv` | `test_0385: 1->0` | 230/394 |
| 4 | `oof/submissions/exp114_drop_test_0013_only_submit.csv` | `test_0013: 1->0` | 230/394 |
| 5 | `oof/submissions/exp114_drop_test_0303_only_submit.csv` | `test_0303: 1->0` | 230/394 |

메모: 계획상 1순위는 `test_0057`이지만, 재계산한 evidence score만 보면 `test_0385`가 가장 높다. `test_0057` 제출이 하락하면 `test_0385`를 바로 다음 후보로 올릴 수 있다.


## EXP-115~116. Public tree update after test_0057 success

### EXP-115 result

- 제출 파일: `oof/submissions/exp114_drop_test_0057_only_submit.csv`
- Public score: `0.9615384615`
- 이전 최고 EXP-110: `0.9599358974`
- 변경: `test_0057: 1 -> 0`
- 해석: `test_0455=0`에 이어 `test_0057=0`도 public에서 이득으로 확인되었다.

현재 유지 방향:

- `test_0455=0`
- `test_0057=0`
- `test_0289=1`

### EXP-116 branch candidates

EXP-114 winner를 새 기준선으로 두고, 남은 후보를 1개씩만 추가 drop한 파일을 생성했다.

| priority | file | change vs EXP-114 | label 0/1 |
|---:|---|---|---:|
| 1 | `oof/submissions/exp116_after0057_drop_test_0385_only_submit.csv` | `test_0385: 1->0` | 231/393 |
| 2 | `oof/submissions/exp116_after0057_drop_test_0172_only_submit.csv` | `test_0172: 1->0` | 231/393 |
| 3 | `oof/submissions/exp116_after0057_drop_test_0013_only_submit.csv` | `test_0013: 1->0` | 231/393 |
| 4 | `oof/submissions/exp116_after0057_drop_test_0303_only_submit.csv` | `test_0303: 1->0` | 231/393 |

다음 제출 1순위는 `exp116_after0057_drop_test_0385_only_submit.csv`다. fold0 ConvNeXt p1이 0.074로 낮고 CAM center fraction도 0.102로 낮아, FP drop 후보로 가장 강하다.


## EXP-116~117. Public tree update after test_0385 success

### EXP-116 result

- 제출 파일: `oof/submissions/exp116_after0057_drop_test_0385_only_submit.csv`
- Public score: `0.9631410256`
- 이전 최고 EXP-114: `0.9615384615`
- 변경: `test_0385: 1 -> 0`
- 해석: `test_0455=0`, `test_0057=0`에 이어 `test_0385=0`도 public에서 이득으로 확인되었다.

현재 유지 방향:

- `test_0455=0`
- `test_0057=0`
- `test_0385=0`
- `test_0289=1`

### EXP-117 branch candidates

EXP-116 winner를 새 기준선으로 두고 남은 후보 3개를 1개씩만 추가 drop한 파일을 생성했다.

| priority | file | change vs EXP-116 | label 0/1 |
|---:|---|---|---:|
| 1 | `oof/submissions/exp117_after0385_drop_test_0172_only_submit.csv` | `test_0172: 1->0` | 232/392 |
| 2 | `oof/submissions/exp117_after0385_drop_test_0013_only_submit.csv` | `test_0013: 1->0` | 232/392 |
| 3 | `oof/submissions/exp117_after0385_drop_test_0303_only_submit.csv` | `test_0303: 1->0` | 232/392 |

다음 제출 1순위는 `exp117_after0385_drop_test_0172_only_submit.csv`다.


## EXP-117 leaderboard result: test_0172 drop failed

- 제출 파일: `oof/submissions/exp117_after0385_drop_test_0172_only_submit.csv`
- Public score: `0.9615384615`
- 현재 최고 EXP-116: `0.9631410256`
- 현재 최고 대비 변경: `test_0172: 1 -> 0`
- 결과: 현재 최고보다 하락

해석: `test_0455=0`, `test_0057=0`, `test_0385=0`은 유지했지만, `test_0172`를 추가로 0으로 바꾸면 public 점수가 하락했다. 따라서 이후 tree에서는 `test_0172=1`을 우선 보호 방향으로 둔다. 남은 제출 후보는 `test_0013` 단독 drop과 `test_0303` 단독 drop이다.


## EXP-117 additional result: test_0013 drop failed

- 제출 파일: `oof/submissions/exp117_after0385_drop_test_0013_only_submit.csv`
- Public score: `0.9615384615`
- 현재 최고 EXP-116: `0.9631410256`
- 현재 최고 대비 변경: `test_0013: 1 -> 0`
- 결과: 현재 최고보다 하락

해석: `test_0455=0`, `test_0057=0`, `test_0385=0`은 유지했지만, `test_0013`를 추가로 0으로 바꾸면 public 점수가 하락했다. 따라서 이후 tree에서는 `test_0013=1`을 우선 보호 방향으로 둔다. 현재 실패 방향은 `test_0289=1`, `test_0172=1`, `test_0013=1`이며, 남은 단일 branch 후보는 `test_0303` 하나다.


## EXP-117 final result: test_0303 drop improved public score

- Submission: `oof/submissions/exp117_after0385_drop_test_0303_only_submit.csv`
- Public score: `0.9647435897`
- Previous best EXP-116: `0.9631410256`
- Change vs EXP-116: `test_0303: 1 -> 0`
- Rank: 7th on 2026-07-01 screenshot
- Remaining submissions: 4

Updated tree guardrails: keep `test_0455=0`, `test_0057=0`, `test_0385=0`, `test_0303=0`; keep failed directions `test_0289=1`, `test_0172=1`, `test_0013=1`. Because only four submissions remain, the next candidate should be selected after re-ranking evidence from the current best instead of blindly continuing the old order.


## EXP-118 remaining submission triage after EXP-117 winner

Current best is `oof/submissions/exp117_after0385_drop_test_0303_only_submit.csv` with public score `0.9647435897` and rank 7 on the 2026-07-01 screenshot. Remaining submissions: 4.

Guardrails after public feedback:
- Success directions: `test_0455=0`, `test_0057=0`, `test_0385=0`, `test_0303=0`
- Failed drop directions: `test_0289=1`, `test_0172=1`, `test_0013=1`

EXP-118 generated four one-sample drop candidates from the current best. Priority 1 is `oof/submissions/exp118_after0303_drop_test_0212_only_submit.csv`, changing only `test_0212: 1 -> 0`. However, these candidates have weaker direct prior/XAI evidence than the successful EXP-110/114/116/117 branches, so the next submission should be evaluated one at a time instead of spending all remaining attempts blindly.


## EXP-119/120 RF-SVM-Gaussian reranking result

Current best remains `oof/submissions/exp117_after0385_drop_test_0303_only_submit.csv` with public score `0.9647435897`. Because only four submissions remain, EXP-119 reranked current-positive samples using OOF model probabilities, RandomForest/ExtraTrees/LinearSVC/Logistic/Ridge meta-model scores, and Gaussian/edge/Laplacian image statistics.

Meta-CV accuracy: RF `0.98735`, ExtraTrees `0.98658`, LinearSVC `0.98601`, Logistic `0.98696`, Ridge `0.98715`.

EXP-119 changed the top candidate from EXP-118 `test_0212` to `test_0547`. EXP-120 generated `oof/submissions/exp120_after0303_drop_test_0547_only_submit.csv`, changing only `test_0547: 1 -> 0` with label distribution 0/1 = 233/391.

Caution: this is a high-risk candidate. `test_0547` ranked high by relative evidence, but absolute class-0 support from RF/ExtraTrees/SVM is not strong. If submitted and it drops, lower the trust in EXP-119 ranking and return to the current-best branch strategy.


## EXP-121/122. RF/SVM/Gaussian evidence ablation consensus

### 목적
EXP-119에서는 RF, SVM, Gaussian evidence를 한 번에 합쳐 `test_0547`을 1순위 후보로 만들었다. 하지만 남은 제출 기회가 4회뿐이므로, 세 근거를 분리해서 어떤 샘플이 반복적으로 상위에 드는지 확인했다.

### 직관
- RF/ExtraTrees: OOF/model probability와 이미지 통계의 비선형 조합을 보는 tabular classifier
- SVM: decision boundary 근처의 선형 분리 신호
- Gaussian evidence: blur/edge/texture 변화 기반 영상 품질 보조 신호

따라서 세 신호를 바로 평균하면 보조 신호가 과대평가될 수 있다. 이번에는 RF-only, SVM-only, Gaussian-only, 2개 조합, 3개 조합을 따로 계산하고 consensus가 높은 샘플을 우선했다.

### 결과
- EXP-121 consensus 1위: `test_0390`
- EXP-119 1위였던 `test_0547`은 evidence 분해 후 우선순위가 내려감
- EXP-122 제출 후보 생성:
  - `oof/submissions/exp122_after0303_consensus_drop_test_0390_only_submit.csv`
  - 현재 최고 `exp117_after0385_drop_test_0303_only_submit.csv` 대비 `test_0390: 1->0` 단일 변경
  - label 분포: 0/1 = 233/391

### 제출 판단
남은 제출 4회 중 하나를 쓴다면 EXP-120 `test_0547`보다 EXP-122 `test_0390`을 먼저 제출하는 쪽이 더 타당하다. 다만 RF/ExtraTrees/SVM이 강하게 class 0이라고 확신한 수준은 아니므로, 제출 후 하락하면 `test_0390=1`을 실패 guardrail로 추가한다.



## EXP-122 제출 결과 반영

- 제출 파일: `oof/submissions/exp122_after0303_consensus_drop_test_0390_only_submit.csv`
- 변경: 현재 최고 대비 `test_0390: 1->0` 1개
- Public score: `0.9631410256`
- 현재 최고 `0.9647435897` 대비 1샘플 하락

### 해석
EXP-121에서 RF/SVM/Gaussian evidence를 분리해 consensus 1위였던 `test_0390`을 제출했지만 하락했다. 따라서 `test_0390=1`을 실패 guardrail로 추가한다. RF/SVM/Gaussian consensus는 단독 제출 결정 근거로는 부족했고, 이후에는 public 성공 방향과 가까운 branch 후보를 우선한다.



## EXP-123. Augmentation and XAI review

### 결론
다음 학습형 실험에서는 좌우반전과 회전을 줄이고, 입력 크기를 `320` 또는 `384`로 키우는 방향이 더 타당하다.

### 근거
기존 학습 스크립트는 대부분 `Resize(224) + RandomHorizontalFlip(0.5) + RandomRotation(5~7)` 구조였다. TTA도 `original + horizontal flip` 중심이었다. 하지만 X-ray에서는 좌우 marker, 심장 위치, 촬영 방향 등 좌우 정보가 완전히 무의미하지 않다. 따라서 horizontal flip을 안전한 기본 증강으로 보기 어렵다.

EXP-112 Grad-CAM 후보를 public 결과와 비교하면, 성공적으로 `1->0` drop된 샘플들은 실패한 keep-1 샘플보다 ConvNeXt fold0 class-1 확률과 CAM center fraction이 낮았다. 즉 모델이 폐 중심 병변 근거를 안정적으로 본 양성이라기보다, 주변/상단/측면 반응에 영향을 받은 후보가 drop 성공과 더 맞았다.

### 다음 transform 후보
```python
train_tf = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
```

보류할 증강은 `RandomHorizontalFlip`, 큰 `RandomRotation`, vertical flip, 강한 crop이다. Grad-CAM은 병변 위치 확정 근거가 아니라 모델 반응 위치를 보는 보조 근거로만 사용한다.



## EXP-124/125: No-Flip 384 ConvNeXt 검증

- 목적: 좌우반전/회전을 제거하고 384 입력 + 약한 brightness/contrast가 ConvNeXt의 오류 방향과 Grad-CAM 근거를 개선하는지 확인합니다.
- EXP-124는 fold0 smoke 실험이며 리더보드 제출용이 아닙니다.
- EXP-125는 EXP-45와 EXP-124를 metric, current-best 변경 샘플, guardrail, Grad-CAM 기준으로 비교합니다.


In [ ]:
# EXP-124 실행 명령
!uv run python scripts/run_exp124_convnext_384_noflip_fold0.py

# EXP-125 비교/XAI 리포트 생성
!uv run python scripts/run_exp125_aug_xai_compare.py


### EXP-124/125 결과 요약

- EXP-124 best epoch: 5
- Fold0 validation accuracy: `0.98755`
- Confusion matrix: `[[265, 4], [9, 766]]`
- Test label count: `0=235`, `1=389`
- 현재 최고 EXP-117 대비 변경 수: `21`
- Guardrail 위반 수: `8`
- 판단: fold0 metric은 개선됐지만 public 방향 확인 샘플을 다수 깨므로 단독 제출 제외, EXP-126 5-fold 확장은 보류합니다.
- 리포트: `reports/exp125_aug_xai_compare/exp125_aug_xai_compare.md`



## EXP-126/127. ConvNeXt 384 No-Flip 5-fold OOF 및 제한적 Stacking

EXP-126에서는 EXP-124 fold0을 재사용하고 fold1~4를 추가 학습해 ConvNeXt-Tiny 384/no-flip 5-fold OOF를 만들었습니다. 목적은 즉시 제출이 아니라, 좌우반전/회전 제거와 384 입력이 새로운 stacking 정보원이 되는지 확인하는 것입니다.

결과적으로 5-fold OOF accuracy는 0.98409였고, 단독 mean submission은 현재 최고 EXP-117 대비 22개가 바뀌며 guardrail 12개를 위반했습니다. 따라서 단독 제출은 제외했습니다.

EXP-127에서는 이 확률을 기존 ConvNeXt/EfficientNet OOF stacking feature에 제한적으로 추가했습니다. 그러나 guardrail을 모두 유지하면서 현재 최고 대비 1~3개만 바뀌는 후보는 0개였습니다. 특히 좋은 meta-CV 조합도 `test_0057`, `test_0385`, `test_0303`을 다시 1로 되돌리는 경향이 있어, 남은 제출 기회는 사용하지 않았습니다.


In [ ]:
# EXP-126/127 요약 확인
import json
from pathlib import Path

for path in [
    Path("oof/preds/exp126_convnext_tiny_384_noflip_combined_result.json"),
    Path("oof/preds/exp127_384_noflip_stacking_result.json"),
]:
    data = json.loads(path.read_text())
    print("\n", path)
    if "oof_metrics" in data:
        print("OOF metrics:", data["oof_metrics"])
        print("test label count:", data["test_mean_pred_counts"])
        print("changed vs EXP-117:", data["current_best_analysis"]["changed_count"])
        print("guardrail violations:", len(data["current_best_analysis"]["guardrail_violations"]))
    else:
        print("candidate count:", data["candidate_count"])
        print("saved submissions:", data["saved"])


## EXP-129/130. Success-branch similarity rerank + Grad-CAM candidate

- 기준 제출: `oof/submissions/exp117_after0385_drop_test_0303_only_submit.csv`
- EXP-128 `test_0600: 1->0` 제출은 public `0.9631410256`으로 하락했으므로 `test_0600=1` 유지 guardrail 추가
- EXP-129: 성공 drop 샘플(`test_0455`, `test_0057`, `test_0385`, `test_0303`)과 닮은 후보를 재랭킹
- EXP-129 top5: `test_0212`, `test_0527`, `test_0267`, `test_0457`, `test_0525`
- EXP-130: EXP-129 top8에 대해 EXP-45 224 ConvNeXt와 EXP-124 384 no-flip ConvNeXt Grad-CAM 비교
- 추천 제출 후보: `oof/submissions/exp130_after0303_successcam_drop_test_0212_submit.csv`
- 현재 최고 대비 변경: `test_0212.png: 1 -> 0`
- 검증: row 624, 컬럼 `file_name,label`, sample submission 순서 일치, label 0/1, label count `0=233`, `1=391`
- 근거: EXP-129 종합 1위, EXP-130 selection 1위, EXP-45 p1 0.3138 / EXP-124 p1 0.3166


## EXP-131. Final submit decision recheck

- 목적: EXP-130 후보 `test_0212: 1->0`을 제출하기 전, `test_0267` 등 대안 후보와 시나리오별 재비교
- 비교 시나리오: balanced, branch similarity, XAI probability, ML evidence, public-branch conservative
- 결과: `test_0212` final score 0.9098, scenario win count 5/5, average rank 1.00
- 대안 `test_0267`은 final score 0.6269, scenario win count 0/5, average rank 2.20
- 추천 제출 파일: `oof/submissions/exp131_final_recheck_drop_test_0212_submit.csv`
- 현재 최고 대비 변경: `test_0212.png: 1 -> 0`
- 검증: row 624, 컬럼 `file_name,label`, sample submission 순서 일치, label count `0=233`, `1=391`


## EXP-132. Success-vs-Failure contrast rerank

- 목적: EXP-131 `test_0212: 1->0` 제출 하락 이후, 성공 drop과 실패 keep을 동시에 반영해 마지막 후보 재정렬
- 성공 drop: `test_0455=0`, `test_0057=0`, `test_0385=0`, `test_0303=0`
- 실패 keep: `test_0289=1`, `test_0172=1`, `test_0013=1`, `test_0390=1`, `test_0600=1`, `test_0212=1` 등
- 방식: 성공 centroid와 가까울수록 가점, 실패 centroid와 가까울수록 감점
- 1순위 후보: `test_0267.png: 1 -> 0`
- 추천 제출 파일: `oof/submissions/exp132_contrast_drop_test_0267_submit.csv`
- 검증: row 624, 컬럼 `file_name,label`, sample submission 순서 일치, label count `0=233`, `1=391`
- 주의: 이 결과는 모델 일반화 개선이 아니라 마지막 제출 후보 선별용임


## 최종 리더보드 결과

- 최종 제출 파일: `oof/submissions/exp132_contrast_drop_test_0267_submit.csv`
- Public score: `0.9663461538`
- 최종 순위: `6위`
- 최종 제출 수: `55`
- 이전 최고 EXP-117 `0.9647435897` 대비 `+0.0016025641`, public 기준 +1샘플 개선
- 성공 방향으로 확인된 변경: `test_0267.png: 1 -> 0`

해석상 주의할 점은, 후반부 EXP-110 이후의 binary-tree/drop branch 실험은 모델의 일반화 성능 자체를 개선한 실험이라기보다 OOF 확률, 모델 disagreement, XAI 요약, public feedback을 결합해 제출 후보를 선별한 전략이라는 점이다. 보고서에서는 ConvNeXt/OOF stacking/XAI 분석을 모델 개발 인사이트로 다루고, 마지막 샘플 단위 drop은 대회 제출 전략으로 분리해서 서술하는 것이 적절하다.
